# DNS Tunneling Detection: Robustness and Transfer Learning Study

**Paper**: *Hardening DNS Tunnel Detection Against Adversarial Mutations and Transfer Learning to Unknown Tools*

This notebook reproduces all experiments from the paper:

| Experiment | Description |
|---|---|
| **Exp 1** | SOTA baseline replication on CIC Bell DNS 2021 |
| **Exp 2** | Adversarial evaluation with ARAGAT mutants |
| **Exp 2.1** | Feature importance and differential robustness analysis |
| **Fig 2** | CNN normalized mean activation profiles |
| **Exp 3** | Ablation study: benign shift vs attack mutation |
| **Exp 4** | Defense hardening (70/30 ARAGAT split) |
| **Exp 5** | Transfer learning to unknown tunneling tools (GraphTunnel) |

**Requirements**: Google Colab + Google Drive with data at `/content/drive/MyDrive/Tunnel/`

## Global Configuration

In [ ]:
import pandas as pd
import joblib
import os
import numpy as np
import glob
import math
import subprocess
import shutil
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# GLOBAL CONFIGURATION
# =============================================================================
# Base directory in Google Drive
BASE_DIR = "/content/drive/MyDrive/Tunnel/CSV_CIC21"

# Directories for Models and Data
MODEL_DIR_SOTA = os.path.join(BASE_DIR, "Models_SOTA_Hybrid")
MODEL_DIR_HARDENED = os.path.join(BASE_DIR, "Models_Hardened")
ARAGAT_DIR = os.path.join(BASE_DIR, "CSV_Generated")
GT_DATA_DIR = "/content/drive/MyDrive/Tunnel/GraphTunnel_Repo" # Usando la ruta persistente recomendada

# Create directories if they don't exist
os.makedirs(MODEL_DIR_SOTA, exist_ok=True)
os.makedirs(MODEL_DIR_HARDENED, exist_ok=True)

# -----------------------------------------------------------------------------
# FEATURE DEFINITIONS (AJUSTADO A TU GENERADOR)
# -----------------------------------------------------------------------------
# Stateless Features (Packet-level)
# Coinciden con tus 'stateless_basic_cols'
FEATS_SL = [
    "FQDN_count", "subdomain_length", "upper", "lower", "numeric",
    "entropy", "special", "labels", "labels_max", "labels_average", "len"
    # Note: 'longest_word' and 'sld' are in the CSV but omitted to keep
    # the feature vector purely numeric and standard.
]

# Stateful Features (Flow-level)
# CRITICAL NOTE: These match EXACTLY with 'stateful_basic_cols' [cite: 92, 105]
FEATS_SF = [
    "rr",
    "A_frequency", "AAAA_frequency", "CNAME_frequency", "TXT_frequency",
    "MX_frequency", "NS_frequency", "NULL_frequency",
    "rr_count", "distinct_ip", "unique_ttl", "total_queries"
]

ALL_FEATS = FEATS_SL + FEATS_SF

print("✅ Environment configured.")
print(f"   Features aligned with Generator: {len(ALL_FEATS)} features total.")
print(f"   Model Directory: {MODEL_DIR_SOTA}")
print(f"   ARAGAT Data Directory: {ARAGAT_DIR}")

## Experiment 1: SOTA Baseline Replication

# 📊 EXPERIMENT 1: BASELINE PERFORMANCE VALIDATION

**Purpose**: Validate that our model implementations match state-of-the-art performance on CIC 2021 benchmark.

**Dataset**: CIC Bell DNS 2021 (140,085 samples total)
- Training set: 112,068 samples (80%)
- Test set: 28,017 samples (20%)
- Split: Stratified (maintains class balance)

**Models Evaluated**: 6 architectures
- Linear: LogisticRegression
- Tree-based: RandomForest, XGBoost, LightGBM
- Deep Learning: CNN, LSTM

**Model Storage**:
Pre-trained baseline models saved to:
```
/content/drive/MyDrive/Tunnel/CSV_CIC21/Models_SOTA_Hybrid/
```

**Results Reported in Table I** (Paper page 5):

| Model | Accuracy | Precision | Recall | F1 |
|-------|----------|-----------|--------|-----|
| LogisticRegression | 93.33% | 88.71% | 98.52% | 0.9336 |
| RandomForest | 93.51% | 88.76% | 98.88% | 0.9355 |
| XGBoost | 93.51% | 88.76% | 98.89% | 0.9355 |
| LightGBM | 93.52% | 88.77% | 98.90% | 0.9356 |
| CNN | 93.49% | 88.76% | 98.83% | 0.9352 |
| LSTM | 93.50% | 88.76% | 98.86% | 0.9354 |

**Validation**: ✅ All models achieve >98.5% recall, matching published state-of-the-art performance on CIC 2021 benchmark.

**Note**: These baseline models are loaded in subsequent experiments (Exp 2-5) for adversarial evaluation against Mutant Payload.

---

In [ ]:
# =============================================================================
# EXPERIMENT 1: SOTA BASELINE REPLICATION (5 MODELS: ML + DL) - OPTIMIZED
# =============================================================================

import os
import glob
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure feature lists exist
if 'FEATS_SL' not in globals() or 'FEATS_SF' not in globals():
    raise ValueError("⚠️ FEATS_SL and FEATS_SF are not defined. Run the Global Configuration cell first.")

# =============================================================================
# CONFIGURATION OPTIMIZADA
# =============================================================================

# Ajuste de rutas
def get_correct_path(subpath):
    if os.path.basename(BASE_DIR) == 'CSV_CIC21' and subpath.startswith('CSV_CIC21'):
        return os.path.join(BASE_DIR, subpath.replace('CSV_CIC21/', '', 1))
    return os.path.join(BASE_DIR, subpath)

CIC_PATHS = {
    'Light_Attack': get_correct_path("CSV_CIC21/CSV_CIC21/Attack_Light_Benign/Attacks"),
    'Light_Benign': get_correct_path("CSV_CIC21/CSV_CIC21/Attack_Light_Benign/Benign"),
    'Heavy_Attack': get_correct_path("CSV_CIC21/CSV_CIC21/Attack_heavy_Benign/Attacks"),
    'Heavy_Benign': get_correct_path("CSV_CIC21/CSV_CIC21/Attack_heavy_Benign/Benign")
}

# =============================================================================
# DATA LOADING FUNCTION (Improved with logging)
# =============================================================================

def load_cic_hybrid_data(verbose=True):
    """Carga datos CIC con logging mejorado"""
    if verbose:
        print("\n🧬 LOADING FULL CIC DATASET (Heavy + Light)...")

    combined_frames = []
    total_files_loaded = 0
    stats = {'attack': 0, 'benign': 0}

    for label_type, folder_path in CIC_PATHS.items():
        if not os.path.exists(folder_path):
            if verbose:
                print(f"   ⚠️ Folder not found: {folder_path}")
            continue

        label = 1 if "Attack" in label_type else 0
        sl_files = glob.glob(os.path.join(folder_path, "stateless_*.csv"))

        if verbose:
            print(f"   📂 Processing {label_type}: {len(sl_files)} files found.")

        for sl_path in sl_files:
            try:
                df_sl = pd.read_csv(sl_path)
                if df_sl.empty:
                    continue

                filename = os.path.basename(sl_path)
                if "light_benign" in filename:
                    sf_filename = filename.replace("stateless_features-light_benign", "stateful_features-_light_benign")
                else:
                    sf_filename = filename.replace("stateless_", "stateful_")

                sf_path = os.path.join(folder_path, sf_filename)

                # Fallback search
                if not os.path.exists(sf_path):
                    candidates = glob.glob(os.path.join(folder_path, "stateful_*.csv"))
                    core_name = filename.replace("stateless_features-", "").replace(".pcap.csv", "")
                    match = next((c for c in candidates if core_name in c), None)
                    if match:
                        sf_path = match
                    else:
                        continue

                df_sf = pd.read_csv(sf_path)
                min_len = min(len(df_sl), len(df_sf))
                if min_len == 0:
                    continue

                df_sl = df_sl.iloc[:min_len].reset_index(drop=True)
                df_sf = df_sf.iloc[:min_len].reset_index(drop=True)

                # Reindex & Fill
                df_sl = df_sl.reindex(columns=FEATS_SL, fill_value=0)
                df_sf = df_sf.reindex(columns=FEATS_SF, fill_value=0)

                for c in FEATS_SL:
                    df_sl[c] = pd.to_numeric(df_sl[c], errors='coerce').fillna(0)
                for c in FEATS_SF:
                    df_sf[c] = pd.to_numeric(df_sf[c], errors='coerce').fillna(0)

                df_hybrid = pd.concat([df_sl[FEATS_SL], df_sf[FEATS_SF]], axis=1)
                df_hybrid['label'] = label
                combined_frames.append(df_hybrid)
                total_files_loaded += 1

                # Statistics
                stats['attack' if label == 1 else 'benign'] += len(df_hybrid)

            except Exception as e:
                if verbose:
                    print(f"      ⚠️ Error loading {os.path.basename(sl_path)}: {e}")

    if not combined_frames:
        raise RuntimeError("❌ CRITICAL: No data loaded.")

    df_final = pd.concat(combined_frames, ignore_index=True)

    if verbose:
        print(f"\n✅ Loaded {total_files_loaded} file pairs")
        print(f"   📊 Attack samples: {stats['attack']:,}")
        print(f"   📊 Benign samples: {stats['benign']:,}")
        print(f"   📊 Total samples: {len(df_final):,}")
        print(f"   📊 Class ratio: {stats['attack']/len(df_final)*100:.1f}% attack")

    return df_final[ALL_FEATS], df_final['label']

# =============================================================================
# MODELOS DEEP LEARNING OPTIMIZADOS
# =============================================================================

def make_cnn_model(input_shape):
    """Improved CNN with more layers and regularization"""
    inputs = keras.Input(shape=input_shape)

    # Block 1
    x = layers.Conv1D(filters=64, kernel_size=3, activation="relu", padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)

    # Block 2
    x = layers.Conv1D(filters=128, kernel_size=3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)

    # Block 3
    x = layers.Conv1D(filters=256, kernel_size=3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    # Dense layers
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="CNN_Optimized")

def make_lstm_model(input_shape):
    """LSTM mejorado con bidireccionalidad"""
    inputs = keras.Input(shape=input_shape)

    # Bidirectional LSTM
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=False))(x)
    x = layers.Dropout(0.3)(x)

    # Dense layers
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="BiLSTM_Optimized")

# =============================================================================
# FULL EVALUATION FUNCTION
# =============================================================================

def evaluate_model(model, X_test, y_test, model_name, is_dl=False):
    """Full evaluation with multiple metrics"""

    if is_dl:
        y_pred_proba = model.predict(X_test, verbose=0).flatten()
        y_pred = (y_pred_proba > 0.5).astype(int)
    else:
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_pred_proba)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # False Positive Rate
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

    print(f"\n   📊 {model_name} Results:")
    print(f"      ├─ Accuracy:  {acc:.4f}")
    print(f"      ├─ Precision: {prec:.4f}")
    print(f"      ├─ Recall:    {rec:.4f}")
    print(f"      ├─ F1-Score:  {f1:.4f}")
    print(f"      ├─ AUC-ROC:   {auc:.4f}")
    print(f"      └─ FPR:       {fpr:.4f}")

    return {
        'model': model_name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'auc_roc': auc,
        'fpr': fpr,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn
    }

# =============================================================================
# OPTIMIZED MAIN EXECUTION
# =============================================================================

try:
    print("\n" + "="*70)
    print("🏆 EXPERIMENT 1: SOTA MODEL TRAINING (OPTIMIZED)")
    print("="*70)

    # A. Carga de Datos
    X_cic, y_cic = load_cic_hybrid_data(verbose=True)

    # B. Calcular class weights para desbalance
    class_weights_array = compute_class_weight('balanced', classes=np.unique(y_cic), y=y_cic)
    class_weights = {i: class_weights_array[i] for i in range(len(class_weights_array))}

    print(f"\n⚖️ Class Weights: {class_weights}")

    # C. Split estratificado
    X_train_cic, X_test_cic, y_train_cic, y_test_cic = train_test_split(
        X_cic, y_cic, test_size=0.2, random_state=42, stratify=y_cic
    )

    print(f"\n📊 Data Split:")
    print(f"   ├─ Train: {len(X_train_cic):,} samples")
    print(f"   │   ├─ Attack: {y_train_cic.sum():,}")
    print(f"   │   └─ Benign: {(y_train_cic == 0).sum():,}")
    print(f"   └─ Test:  {len(X_test_cic):,} samples")
    print(f"       ├─ Attack: {y_test_cic.sum():,}")
    print(f"       └─ Benign: {(y_test_cic == 0).sum():,}")

    # D. Preprocesamiento DL
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_cic)
    X_test_scaled = scaler.transform(X_test_cic)

    # Guardar scaler
    os.makedirs(MODEL_DIR_SOTA, exist_ok=True)
    joblib.dump(scaler, os.path.join(MODEL_DIR_SOTA, "scaler_sota.joblib"))

    # Reshape para DL
    X_train_dl = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_test_dl = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))
    input_shape = (X_train_dl.shape[1], 1)

    # ==========================================================================
    # GROUP 1: CLASSICAL ML MODELS (OPTIMIZED)
    # ==========================================================================

    print("\n" + "="*70)
    print("🤖 TRAINING CLASSICAL ML MODELS (Optimized Hyperparameters)")
    print("="*70)

    ML_MODELS = {
        'LogisticRegression': LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            solver='lbfgs',
            n_jobs=-1,
            random_state=42
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=300,           # More trees
            max_depth=20,               # Profundidad controlada
            min_samples_split=5,        # Evitar overfitting
            min_samples_leaf=2,
            class_weight='balanced',    # Automatic class balancing
            n_jobs=-1,
            random_state=42
        ),
        'XGBoost': XGBClassifier(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.05,         # Lower learning rate
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=class_weights[1]/class_weights[0],  # Balance
            use_label_encoder=False,
            eval_metric='logloss',
            n_jobs=-1,
            random_state=42
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight='balanced',
            n_jobs=-1,
            verbose=-1,
            random_state=42
        )
    }

    ml_results = []

    for name, clf in ML_MODELS.items():
        print(f"\n⚙️ Training {name}...")

        # Para Logistic Regression usar datos escalados
        if name == 'LogisticRegression':
            clf.fit(X_train_scaled, y_train_cic)
            result = evaluate_model(clf, X_test_scaled, y_test_cic, name, is_dl=False)
        else:
            clf.fit(X_train_cic, y_train_cic)
            result = evaluate_model(clf, X_test_cic, y_test_cic, name, is_dl=False)

        # Guardar modelo
        joblib.dump(clf, os.path.join(MODEL_DIR_SOTA, f"{name}_sota.joblib"))

        ml_results.append(result)

        # Feature importance (top 10) - solo para modelos con esta capacidad
        if hasattr(clf, 'feature_importances_'):
            importances = clf.feature_importances_
            indices = np.argsort(importances)[::-1][:10]
            print(f"\n      🔍 Top 10 Features:")
            for i, idx in enumerate(indices, 1):
                feat_name = ALL_FEATS[idx]
                print(f"         {i}. {feat_name}: {importances[idx]:.4f}")
        elif hasattr(clf, 'coef_'):
            # For Logistic Regression, show most important coefficients
            coef = np.abs(clf.coef_[0])
            indices = np.argsort(coef)[::-1][:10]
            print(f"\n      🔍 Top 10 Features (by coefficient magnitude):")
            for i, idx in enumerate(indices, 1):
                feat_name = ALL_FEATS[idx]
                print(f"         {i}. {feat_name}: {coef[idx]:.4f}")

    # ==========================================================================
    # GRUPO 2: MODELOS DEEP LEARNING - OPTIMIZADOS
    # ==========================================================================

    print("\n" + "="*70)
    print("🧠 TRAINING DEEP LEARNING MODELS (Optimized Architecture)")
    print("="*70)

    DL_CONFIG = {
        'LSTM': make_lstm_model(input_shape),
        'CNN': make_cnn_model(input_shape)
    }

    dl_results = []

    for name, model in DL_CONFIG.items():
        print(f"\n⚙️ Training {name}...")

        # Configurar callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                verbose=1
            ),
            ModelCheckpoint(
                filepath=os.path.join(MODEL_DIR_SOTA, f"{name}_sota_best.keras"),
                monitor='val_loss',
                save_best_only=True,
                verbose=0
            )
        ]

        # Compilar con class weights
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='binary_crossentropy',
            metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Recall(name='recall')]
        )

        # Train for 20 epochs
        history = model.fit(
            X_train_dl, y_train_cic,
            epochs=20,                  # Reduced to 20 epochs
            batch_size=64,
            validation_split=0.15,
            class_weight=class_weights,
            callbacks=callbacks,
            verbose=1
        )

        # Guardar modelo final
        model.save(os.path.join(MODEL_DIR_SOTA, f"{name}_sota.keras"))

        # Evaluar
        result = evaluate_model(model, X_test_dl, y_test_cic, name, is_dl=True)
        dl_results.append(result)

    # ==========================================================================
    # RESULTADOS FINALES
    # ==========================================================================

    print("\n" + "="*70)
    print("📊 FINAL RESULTS SUMMARY")
    print("="*70)

    all_results = ml_results + dl_results
    results_df = pd.DataFrame(all_results)

    # Ordenar por F1-Score
    results_df = results_df.sort_values('f1_score', ascending=False)

    print("\n" + results_df.to_string(index=False))

    # Guardar resultados
    results_df.to_csv(os.path.join(MODEL_DIR_SOTA, "experiment1_results.csv"), index=False)

    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # 1. Metrics comparison
    metrics = ['accuracy', 'precision', 'recall', 'f1_score']
    results_df[['model'] + metrics].set_index('model').plot(kind='bar', ax=axes[0, 0])
    axes[0, 0].set_title('Model Performance Comparison')
    axes[0, 0].set_ylabel('Score')
    axes[0, 0].legend(loc='lower right')
    axes[0, 0].grid(True, alpha=0.3)

    # 2. AUC-ROC Comparison
    results_df[['model', 'auc_roc']].set_index('model').plot(kind='barh', ax=axes[0, 1], legend=False)
    axes[0, 1].set_title('AUC-ROC Comparison')
    axes[0, 1].set_xlabel('AUC-ROC Score')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. False Positive Rate
    results_df[['model', 'fpr']].set_index('model').plot(kind='barh', ax=axes[1, 0], legend=False, color='red')
    axes[1, 0].set_title('False Positive Rate (Lower is Better)')
    axes[1, 0].set_xlabel('FPR')
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Confusion Matrix del mejor modelo
    best_model_name = results_df.iloc[0]['model']
    cm_data = results_df.iloc[0][['tn', 'fp', 'fn', 'tp']].values.reshape(2, 2)
    sns.heatmap(cm_data, annot=True, fmt='g', cmap='Blues', ax=axes[1, 1])
    axes[1, 1].set_title(f'Confusion Matrix - Best Model ({best_model_name})')
    axes[1, 1].set_xlabel('Predicted')
    axes[1, 1].set_ylabel('Actual')

    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_DIR_SOTA, "experiment1_results.png"), dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\n✅ Experiment 1 Complete!")
    print(f"   📁 Models saved to: {MODEL_DIR_SOTA}")
    print(f"   📊 Results saved to: experiment1_results.csv")
    print(f"   📈 Visualization saved to: experiment1_results.png")

    print(f"\n🏆 Best Model: {best_model_name}")
    print(f"   F1-Score: {results_df.iloc[0]['f1_score']:.4f}")
    print(f"   Recall: {results_df.iloc[0]['recall']:.4f}")

except RuntimeError as e:
    print(f"\n❌ CRITICAL ERROR: {e}")
except Exception as e:
    print(f"\n❌ Unexpected Error: {e}")
    import traceback
    traceback.print_exc()

## Experiment 2: Adversarial Evaluation (ARAGAT Mutants)

Evaluate all baseline models against ARAGAT-generated DNS tunnel mutant payloads.
Mutants are created by systematically varying encoding parameters across 9 random seeds.

In [ ]:
# =============================================================================
# EXPERIMENT 2: ADVERSARIAL EVALUATION (DYNAMIC SEEDS)
# Automatically detects all available seeds in the directory
# =============================================================================

import os
import re
import pandas as pd
import numpy as np
import joblib
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    confusion_matrix
)

# =============================================================================
# CONFIGURATION
# =============================================================================

ARAGAT_DIR = "/content/drive/MyDrive/Tunnel/CSV_CIC21/CSV_Generated"
MODEL_DIR_SOTA = "/content/drive/MyDrive/Tunnel/CSV_CIC21/Models_SOTA_Hybrid"

# Features (las mismas 23 que usaste para entrenar)
FEATS_SL = [
    "FQDN_count", "subdomain_length", "upper", "lower", "numeric",
    "entropy", "special", "labels", "labels_max", "labels_average", "len"
]

FEATS_SF = [
    "rr",
    "A_frequency", "AAAA_frequency", "CNAME_frequency", "TXT_frequency",
    "MX_frequency", "NS_frequency", "NULL_frequency",
    "rr_count", "distinct_ip", "unique_ttl", "total_queries"
]

ALL_FEATS = FEATS_SL + FEATS_SF

# =============================================================================
# FUNCTION TO AUTO-DETECT SEEDS
# =============================================================================

def detect_available_seeds(directory, verbose=True):
    """
    Escanea el directorio y detecta todos los seeds disponibles

    Searches files matching pattern: stateless_features-bridge.pcap_[SEED].csv

    Args:
        directory: Ruta del directorio a escanear
        verbose: Show search information

    Returns:
        Lista de seeds encontrados (ordenados)
    """
    if verbose:
        print(f"\n🔍 Scanning directory: {directory}")

    if not os.path.exists(directory):
        raise FileNotFoundError(f"Directory not found: {directory}")

    # Pattern para detectar seeds en archivos stateless
    pattern = r'stateless_features-bridge\.pcap_(\d+)\.csv'

    seeds_found = set()

    for filename in os.listdir(directory):
        match = re.search(pattern, filename)
        if match:
            seed = match.group(1)

            # Verify that the corresponding stateful file also exists
            stateful_file = f"stateful_features-bridge.pcap_{seed}.csv"
            stateful_path = os.path.join(directory, stateful_file)

            if os.path.exists(stateful_path):
                seeds_found.add(seed)
                if verbose:
                    print(f"   ✅ Found seed {seed} (stateless + stateful)")
            else:
                if verbose:
                    print(f"   ⚠️  Found seed {seed} but missing stateful file")

    seeds_sorted = sorted(seeds_found, key=lambda x: int(x))

    if verbose:
        print(f"\n📊 Total seeds found: {len(seeds_sorted)}")
        if seeds_sorted:
            print(f"   Seeds: {', '.join(seeds_sorted)}")

    return seeds_sorted

# =============================================================================
# FUNCTION TO PROCESS A SINGLE SEED
# =============================================================================

def process_single_seed(directory, seed, verbose=True):
    """
    Processes stateless and stateful files for a specific seed

    Args:
        directory: Directorio base
        seed: Seed number (string or int)
        verbose: Mostrar mensajes de progreso

    Returns:
        DataFrame con features alineadas y labels
    """
    seed_str = str(seed)

    if verbose:
        print(f"\n   📂 Processing seed {seed_str}...")

    # Construir rutas
    stateless_path = os.path.join(directory, f"stateless_features-bridge.pcap_{seed_str}.csv")
    stateful_path = os.path.join(directory, f"stateful_features-bridge.pcap_{seed_str}.csv")

    # Validar archivos
    if not os.path.exists(stateless_path):
        raise FileNotFoundError(f"❌ Stateless file not found: {stateless_path}")
    if not os.path.exists(stateful_path):
        raise FileNotFoundError(f"❌ Stateful file not found: {stateful_path}")

    # Cargar datos
    df_sl = pd.read_csv(stateless_path)
    df_sf = pd.read_csv(stateful_path)

    if verbose:
        print(f"      ├─ Stateless: {len(df_sl):,} rows")
        print(f"      └─ Stateful:  {len(df_sf):,} rows")

    # Limpiar stateless
    for c in FEATS_SL:
        if c in df_sl.columns:
            df_sl[c] = pd.to_numeric(df_sl[c], errors='coerce')
        else:
            df_sl[c] = 0

    # Agregar por tunnel_id (query-level → flow-level)
    aggregation_dict = {feat: 'mean' for feat in FEATS_SL if feat in df_sl.columns}
    if 'label' in df_sl.columns:
        aggregation_dict['label'] = 'first'

    df_sl_agg = df_sl.groupby('tunnel_id').agg(aggregation_dict).reset_index()

    # CRITICAL: Add prefix to tunnel_id to avoid collisions across seeds
    df_sl_agg['tunnel_id'] = df_sl_agg['tunnel_id'].astype(str) + f"_s{seed_str}"

    if verbose:
        print(f"      ├─ Aggregated to {len(df_sl_agg):,} flows")

    # Limpiar stateful
    for c in FEATS_SF:
        if c in df_sf.columns:
            df_sf[c] = pd.to_numeric(df_sf[c], errors='coerce')
        else:
            df_sf[c] = 0

    # CRITICAL: Also add prefix to stateful tunnel_id
    df_sf['tunnel_id'] = df_sf['tunnel_id'].astype(str) + f"_s{seed_str}"

    # Merge stateless + stateful
    df_merged = pd.merge(df_sl_agg, df_sf, on='tunnel_id', how='inner', suffixes=('', '_sf'))

    if verbose:
        print(f"      └─ Merged: {len(df_merged):,} flows")

    # Alinear columnas con CIC-2021 training
    df_merged = df_merged.reindex(columns=ALL_FEATS + ['label', 'tunnel_id'], fill_value=0)

    # Procesar labels
    if 'label' in df_merged.columns:
        df_merged['label'] = df_merged['label'].fillna(1).astype(int)
    else:
        df_merged['label'] = 1  # Asumir ataque si no hay label

    # Rellenar NaN/inf
    df_merged[ALL_FEATS] = df_merged[ALL_FEATS].replace([np.inf, -np.inf], np.nan).fillna(0)

    # Add seed column
    df_merged['seed'] = int(seed_str)

    return df_merged

# =============================================================================
# MAIN FUNCTION: LOAD ALL AVAILABLE SEEDS
# =============================================================================

def load_aragat_all_seeds(directory=ARAGAT_DIR, specific_seeds=None, verbose=True):
    """
    Carga dataset completo con TODOS los seeds disponibles

    Args:
        directory: Directory containing the CSVs
        specific_seeds: List of specific seeds to load (None = all)
        verbose: Show progress information

    Returns:
        X: Features
        y: Labels
        tunnel_ids: Unique IDs with seed prefix
        seed_labels: Array indicando el seed de cada flow
        seeds_loaded: Lista de seeds que se cargaron
    """
    if verbose:
        print("\n" + "="*70)
        print("💀 LOADING ARAGAT/MUTANT-DNS DATASET (DYNAMIC SEEDS)")
        print("="*70)

    # Detectar seeds disponibles
    available_seeds = detect_available_seeds(directory, verbose=verbose)

    if not available_seeds:
        raise RuntimeError("❌ No valid seeds found in directory")

    # Determine which seeds to load
    if specific_seeds is not None:
        seeds_to_load = [str(s) for s in specific_seeds if str(s) in available_seeds]
        missing = [str(s) for s in specific_seeds if str(s) not in available_seeds]

        if missing:
            print(f"\n⚠️  WARNING: Requested seeds not found: {', '.join(missing)}")

        if not seeds_to_load:
            raise RuntimeError("❌ None of the requested seeds are available")
    else:
        seeds_to_load = available_seeds

    if verbose:
        print(f"\n📥 Loading {len(seeds_to_load)} seed(s): {', '.join(seeds_to_load)}")

    # Cargar todos los seeds
    dfs = []

    for seed in seeds_to_load:
        try:
            df_seed = process_single_seed(directory, seed, verbose=verbose)
            dfs.append(df_seed)
        except Exception as e:
            print(f"\n❌ Error loading seed {seed}: {e}")
            if verbose:
                import traceback
                traceback.print_exc()
            continue

    if not dfs:
        raise RuntimeError("❌ No seeds were successfully loaded")

    # Combinar todos los seeds
    if verbose:
        print(f"\n   🔗 Combining all seeds...")

    df_combined = pd.concat(dfs, ignore_index=True)

    if verbose:
        print(f"      ✅ Total combined: {len(df_combined):,} flows")
        for i, seed in enumerate(seeds_to_load):
            print(f"         {'└─' if i == len(seeds_to_load)-1 else '├─'} Seed {seed}: {len(dfs[i]):,} flows")

    # Validaciones
    if verbose:
        print(f"\n   🔍 Validating data integrity...")

    # Verificar que no hay tunnel_ids duplicados
    n_unique = df_combined['tunnel_id'].nunique()
    n_total = len(df_combined)

    if n_unique != n_total:
        print(f"      ⚠️  WARNING: {n_total - n_unique} duplicate tunnel_ids found!")
    else:
        print(f"      ✅ All {n_total:,} tunnel_ids are unique")

    # Verificar features sin NaN/inf
    n_invalid = df_combined[ALL_FEATS].isna().sum().sum()
    n_invalid += np.isinf(df_combined[ALL_FEATS]).sum().sum()

    if n_invalid > 0:
        print(f"      ⚠️  WARNING: {n_invalid} invalid values (NaN/inf) found!")
    else:
        print(f"      ✅ No invalid values in features")

    # Statistics finales
    if verbose:
        print(f"\n   📊 Dataset Statistics:")

        print(f"\n      Overall:")
        n_attack = (df_combined['label'] == 1).sum()
        n_benign = (df_combined['label'] == 0).sum()
        print(f"      ├─ Attack flows:  {n_attack:,} ({n_attack/len(df_combined)*100:.1f}%)")
        print(f"      └─ Benign flows:  {n_benign:,} ({n_benign/len(df_combined)*100:.1f}%)")

        print(f"\n      By Seed:")
        for i, seed in enumerate(seeds_to_load):
            mask = df_combined['seed'] == int(seed)
            n_total_seed = mask.sum()
            n_attack_seed = ((df_combined['seed'] == int(seed)) & (df_combined['label'] == 1)).sum()
            n_benign_seed = ((df_combined['seed'] == int(seed)) & (df_combined['label'] == 0)).sum()

            prefix = '└─' if i == len(seeds_to_load)-1 else '├─'
            print(f"      {prefix} Seed {seed}: {n_total_seed:,} flows")
            print(f"         ├─ Attack:  {n_attack_seed:,} ({n_attack_seed/n_total_seed*100:.1f}%)")
            print(f"         └─ Benign:  {n_benign_seed:,} ({n_benign_seed/n_total_seed*100:.1f}%)")

    # Retornar datos
    X = df_combined[ALL_FEATS].values
    y = df_combined['label'].values
    tunnel_ids = df_combined['tunnel_id'].values
    seed_labels = df_combined['seed'].values

    return X, y, tunnel_ids, seed_labels, seeds_to_load

# =============================================================================
# EVALUATION FUNCTION
# =============================================================================

def evaluate_model_adversarial(model, X_test, y_test, model_name, is_dl=False, scaler=None):
    """
    Evaluates a model on the adversarial set
    """
    # Prepare data according to model type
    if model_name == 'LogisticRegression':
        if scaler is None:
            raise ValueError("Scaler required for LogisticRegression")
        X_eval = scaler.transform(X_test)
        y_pred = model.predict(X_eval)
        y_proba = model.predict_proba(X_eval)[:, 1]

    elif is_dl:  # CNN o LSTM
        if scaler is None:
            raise ValueError("Scaler required for DL models")
        X_scaled = scaler.transform(X_test)
        X_dl = X_scaled.reshape((X_scaled.shape[0], X_scaled.shape[1], 1))
        y_proba = model.predict(X_dl, verbose=0).flatten()
        y_pred = (y_proba > 0.5).astype(int)

    else:  # Tree-based models
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else y_pred.astype(float)

    # Compute metrics
    recall = recall_score(y_test, y_pred, zero_division=0)
    precision = precision_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    accuracy = accuracy_score(y_test, y_pred)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    # Handle different confusion matrix sizes
    if cm.shape == (1, 1):
        if y_test[0] == 1:  # Solo ataques
            tp = cm[0, 0] if y_pred[0] == 1 else 0
            fn = cm[0, 0] if y_pred[0] == 0 else 0
            fp = tn = 0
        else:
            tn = cm[0, 0] if y_pred[0] == 0 else 0
            fp = cm[0, 0] if y_pred[0] == 1 else 0
            tp = fn = 0
    elif cm.shape == (1, 2):
        if y_test[0] == 1:
            fn, tp = cm[0]
            fp = tn = 0
        else:
            tn, fp = cm[0]
            tp = fn = 0
    elif cm.shape == (2, 1):
        if y_pred[0] == 1:
            fn, tp = cm[:, 0]
            tn = fp = 0
        else:
            tn, fp = cm[:, 0]
            tp = fn = 0
    else:
        tn, fp, fn, tp = cm.ravel()

    evasion_rate = 1 - recall

    return {
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'accuracy': accuracy,
        'evasion_rate': evasion_rate,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    try:
        print("\n" + "="*70)
        print("⚔️  EXPERIMENT 2: ADVERSARIAL ROBUSTNESS (DYNAMIC SEEDS)")
        print("="*70)

        # =====================================================================
        # CONFIGURATION: Specify particular seeds or load all available
        # =====================================================================

        # Option 1: Load ALL available seeds
        SPECIFIC_SEEDS = None

        # Option 2: Load only specific seeds (uncomment to use)
        # SPECIFIC_SEEDS = [42, 75, 99]  # Solo estos seeds
        # SPECIFIC_SEEDS = [42, 75, 99, 123, 456]  # More seeds

        # =====================================================================
        # 1. LOAD DATA (AUTOMATIC)
        # =====================================================================
        X_aragat, y_aragat, tunnel_ids, seed_labels, seeds_loaded = load_aragat_all_seeds(
            directory=ARAGAT_DIR,
            specific_seeds=SPECIFIC_SEEDS,
            verbose=True
        )

        print(f"\n✅ Data loaded successfully!")
        print(f"   ├─ Shape: {X_aragat.shape}")
        print(f"   ├─ Features: {X_aragat.shape[1]}")
        print(f"   ├─ Flows: {X_aragat.shape[0]:,}")
        print(f"   └─ Seeds: {len(seeds_loaded)} ({', '.join(seeds_loaded)})")

        # =====================================================================
        # 2. CARGAR SCALER
        # =====================================================================
        scaler_path = os.path.join(MODEL_DIR_SOTA, "scaler_sota.joblib")

        if not os.path.exists(scaler_path):
            raise FileNotFoundError(f"❌ Scaler not found: {scaler_path}")

        scaler = joblib.load(scaler_path)
        print(f"\n✅ Scaler loaded from: {scaler_path}")

        # =====================================================================
        # 3. EVALUAR TODOS LOS MODELOS
        # =====================================================================
        MODEL_NAMES = ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CNN', 'LSTM']

        results = []

        print("\n" + "="*70)
        print(f"🔍 EVALUATING MODELS ON ADVERSARIAL FLOWS ({len(seeds_loaded)} SEEDS)")
        print("="*70)

        for model_name in MODEL_NAMES:
            print(f"\n⚙️  Evaluating {model_name}...")

            try:
                # Determinar ruta del modelo
                if model_name in ['CNN', 'LSTM']:
                    model_path = os.path.join(MODEL_DIR_SOTA, f"{model_name}_sota.keras")
                    is_dl = True
                else:
                    model_path = os.path.join(MODEL_DIR_SOTA, f"{model_name}_sota.joblib")
                    is_dl = False

                # Validar archivo
                if not os.path.exists(model_path):
                    print(f"   ⚠️  Model file not found: {model_path}")
                    continue

                # Cargar modelo
                if is_dl:
                    model = keras.models.load_model(model_path)
                else:
                    model = joblib.load(model_path)

                # Evaluar en TODOS los flows
                result_total = evaluate_model_adversarial(
                    model, X_aragat, y_aragat, model_name, is_dl=is_dl, scaler=scaler
                )

                # Evaluar por SEED (para validar consistencia)
                seed_results = []

                for seed in seeds_loaded:
                    mask = seed_labels == int(seed)

                    if mask.sum() > 0:
                        result_seed = evaluate_model_adversarial(
                            model, X_aragat[mask], y_aragat[mask], model_name, is_dl=is_dl, scaler=scaler
                        )
                        seed_results.append({
                            'seed': seed,
                            'recall': result_seed['recall'],
                            'n_flows': mask.sum()
                        })

                # Calcular consistencia entre seeds
                recalls = [r['recall'] for r in seed_results]
                consistency_std = np.std(recalls) if len(recalls) > 1 else 0.0
                consistency_range = (max(recalls) - min(recalls)) if len(recalls) > 1 else 0.0

                # Mostrar resultados
                print(f"   📊 Results:")
                print(f"      Overall ({X_aragat.shape[0]:,} flows):")
                print(f"      ├─ Recall:        {result_total['recall']:.2%}")
                print(f"      ├─ Precision:     {result_total['precision']:.2%}")
                print(f"      ├─ Evasion Rate:  {result_total['evasion_rate']:.2%}")
                print(f"      └─ F1-Score:      {result_total['f1']:.4f}")

                print(f"\n      By Seed:")
                for i, sr in enumerate(seed_results):
                    prefix = '└─' if i == len(seed_results)-1 else '├─'
                    print(f"      {prefix} Seed {sr['seed']}: {sr['recall']:.2%} recall ({sr['n_flows']:,} flows)")

                if len(seeds_loaded) > 1:
                    print(f"      ├─ Std Dev:  {consistency_std:.4f}")
                    print(f"      └─ Range:    {consistency_range:.2%}")

                    if consistency_range < 0.03:
                        status = "✅ Excellent consistency"
                    elif consistency_range < 0.07:
                        status = "🟢 Good consistency"
                    elif consistency_range < 0.15:
                        status = "🟡 Moderate consistency"
                    else:
                        status = "⚠️  High variance"

                    print(f"      Status: {status}")

                # Guardar resultados
                result_dict = {
                    'model': model_name,
                    'recall_total': result_total['recall'],
                    'precision_total': result_total['precision'],
                    'f1_total': result_total['f1'],
                    'evasion_total': result_total['evasion_rate'],
                    'consistency_std': consistency_std,
                    'consistency_range': consistency_range,
                    'tp': result_total['tp'],
                    'fp': result_total['fp'],
                    'tn': result_total['tn'],
                    'fn': result_total['fn'],
                    'n_seeds': len(seeds_loaded),
                    'seeds': ','.join(seeds_loaded)
                }

                # Add per-seed recall
                for sr in seed_results:
                    result_dict[f"recall_seed{sr['seed']}"] = sr['recall']

                results.append(result_dict)

            except Exception as e:
                print(f"   ❌ Error: {e}")
                import traceback
                traceback.print_exc()

        # =====================================================================
        # 4. GUARDAR RESULTADOS
        # =====================================================================
        if not results:
            raise RuntimeError("❌ No models were successfully evaluated")

        df_results = pd.DataFrame(results)

        # Ordenar por evasion rate (de mayor a menor)
        df_results = df_results.sort_values('evasion_total', ascending=False)

        # Guardar CSV
        seeds_suffix = '_'.join(seeds_loaded[:3]) + (f'_plus{len(seeds_loaded)-3}' if len(seeds_loaded) > 3 else '')
        output_path = os.path.join(MODEL_DIR_SOTA, f"experiment2_seeds_{seeds_suffix}_results.csv")
        df_results.to_csv(output_path, index=False)

        print("\n" + "="*70)
        print("📊 SUMMARY RESULTS")
        print("="*70)

        # Columns to display (dynamic based on available seeds)
        base_cols = ['model', 'recall_total', 'evasion_total']
        seed_cols = [col for col in df_results.columns if col.startswith('recall_seed')]
        display_cols = base_cols + seed_cols[:5] + ['consistency_range']  # Mostrar hasta 5 seeds

        print("\n" + df_results[display_cols].to_string(index=False))

        # =====================================================================
        # 5. STATISTICAL ANALYSIS
        # =====================================================================
        print("\n" + "="*70)
        print("📈 STATISTICAL SUMMARY")
        print("="*70)

        print(f"\n🎯 Evasion Rates:")
        print(f"   ├─ Mean:   {df_results['evasion_total'].mean():.2%}")
        print(f"   ├─ Median: {df_results['evasion_total'].median():.2%}")
        print(f"   ├─ Min:    {df_results['evasion_total'].min():.2%}")
        print(f"   └─ Max:    {df_results['evasion_total'].max():.2%}")

        print(f"\n🔍 Detection Recalls:")
        print(f"   ├─ Mean:   {df_results['recall_total'].mean():.2%}")
        print(f"   ├─ Median: {df_results['recall_total'].median():.2%}")
        print(f"   ├─ Min:    {df_results['recall_total'].min():.2%}")
        print(f"   └─ Max:    {df_results['recall_total'].max():.2%}")

        if len(seeds_loaded) > 1:
            print(f"\n🎲 Cross-Seed Consistency:")
            print(f"   ├─ Mean std dev:   {df_results['consistency_std'].mean():.4f}")
            print(f"   ├─ Median std dev: {df_results['consistency_std'].median():.4f}")
            print(f"   ├─ Mean range:     {df_results['consistency_range'].mean():.2%}")
            print(f"   └─ Max range:      {df_results['consistency_range'].max():.2%}")

        # Critical failures
        critical = (df_results['recall_total'] < 0.1).sum()
        print(f"\n⚠️  Critical Failures (Recall < 10%): {critical}/{len(df_results)}")

        # =====================================================================
        # 6. CONCLUSION
        # =====================================================================
        print("\n" + "="*70)
        print(f"✅ EXPERIMENT 2 COMPLETE ({len(seeds_loaded)} SEEDS)")
        print("="*70)

        print(f"\n📁 Results saved to:")
        print(f"   └─ {output_path}")

        avg_evasion = df_results['evasion_total'].mean()

        if avg_evasion > 0.8:
            conclusion = "🔴 CRITICAL: Average evasion >80%. Models are highly vulnerable."
        elif avg_evasion > 0.5:
            conclusion = "🟠 HIGH: Average evasion >50%. Significant vulnerability."
        elif avg_evasion > 0.3:
            conclusion = "🟡 MODERATE: Average evasion >30%. Partial vulnerability."
        else:
            conclusion = "🟢 LOW: Models show good robustness."

        print(f"\n💡 Conclusion: {conclusion}")

    except Exception as e:
        print(f"\n❌ CRITICAL ERROR: {e}")
        import traceback
        traceback.print_exc()

## Experiment 2.1: Feature Importance and Differential Robustness

Analyze which features drive detection and which models are most robust to adversarial mutations.

In [ ]:
# =============================================================================
# IN-DEPTH ANALYSIS: FEATURE IMPORTANCE AND DIFFERENTIAL ROBUSTNESS (TODOS LOS MODELOS)
# VERSION V2 — 4 fixes over the original notebook:
#   1. RF with real std (among individual trees)
#   2. CSV consolidado: 1 fila por feature × columnas por modelo (importance + std)
#   3. Columna 'category' exportada: Positional / Stateful / Other
#   4. recall_score real (clase ataque) en lugar de accuracy global
#   5. Barras de error (xerr=std) en figura para RF, CNN, LSTM
# =============================================================================

import os
import re
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator
from sklearn.metrics import recall_score          # ← FIX 4
from tensorflow import keras

# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_DIR  = "/content/drive/MyDrive/Tunnel/CSV_CIC21/Models_SOTA_Hybrid"
ARAGAT_DIR = "/content/drive/MyDrive/Tunnel/CSV_CIC21/CSV_Generated"

FEATS_SL = [
    "FQDN_count", "subdomain_length", "upper", "lower", "numeric",
    "entropy", "special", "labels", "labels_max", "labels_average", "len"
]
FEATS_SF = [
    "rr",
    "A_frequency", "AAAA_frequency", "CNAME_frequency", "TXT_frequency",
    "MX_frequency", "NS_frequency", "NULL_frequency",
    "rr_count", "distinct_ip", "unique_ttl", "total_queries"
]
ALL_FEATS = FEATS_SL + FEATS_SF   # 23 features

# =============================================================================
# FIX 3 — Feature categories (defined globally for export)
# =============================================================================

# Positional: depend on content/position of the DNS name payload
POSITIONAL_FEATS = [
    'FQDN_count', 'subdomain_length', 'upper', 'lower', 'numeric',
    'entropy', 'special', 'labels', 'labels_max', 'labels_average', 'len'
]
# Stateful: flow statistics, more invariant to position
STATEFUL_FEATS = [
    'rr', 'A_frequency', 'AAAA_frequency', 'CNAME_frequency', 'TXT_frequency',
    'MX_frequency', 'NS_frequency', 'NULL_frequency',
    'rr_count', 'distinct_ip', 'unique_ttl', 'total_queries'
]

def assign_category(feature_name):
    if feature_name in POSITIONAL_FEATS:
        return 'Positional'
    elif feature_name in STATEFUL_FEATS:
        return 'Stateful'
    return 'Other'

# =============================================================================
# FUNCTION TO AUTO-DETECT SEEDS
# =============================================================================

def detect_available_seeds(directory, verbose=True):
    if verbose:
        print(f"\n🔍 Scanning directory: {directory}")
    if not os.path.exists(directory):
        raise FileNotFoundError(f"❌ Directory not found: {directory}")
    files = os.listdir(directory)
    pattern = re.compile(r'stateless_features-bridge\.pcap_(\d+)\.csv')
    seeds_found = set()
    for filename in files:
        match = pattern.match(filename)
        if match:
            seed = int(match.group(1))
            if f"stateful_features-bridge.pcap_{seed}.csv" in files:
                seeds_found.add(seed)
                if verbose:
                    try:
                        df_temp = pd.read_csv(os.path.join(directory, filename))
                        n_attack = (df_temp['label'] == 1).sum() if 'label' in df_temp.columns else 0
                        print(f"   ✅ Found seed {seed}: {len(df_temp):,} flows ({n_attack} attack)")
                    except Exception:
                        print(f"   ✅ Found seed {seed}")
            elif verbose:
                print(f"   ⚠️  Seed {seed}: stateless found but stateful missing")
    seeds_sorted = sorted(list(seeds_found))
    if verbose:
        print(f"\n📊 Total seeds found: {len(seeds_sorted)}")
        if seeds_sorted:
            print(f"   Seeds: {', '.join(map(str, seeds_sorted))}")
    return seeds_sorted

# =============================================================================
# FUNCTION TO LOAD DATA (DYNAMIC SEEDS)
# =============================================================================

def process_single_seed(stateless_path, stateful_path, seed_name, verbose=True):
    if verbose:
        print(f"\n   📂 Processing seed {seed_name}...")
    df_sl = pd.read_csv(stateless_path)
    df_sf = pd.read_csv(stateful_path)
    if verbose:
        print(f"      ├─ Stateless: {len(df_sl):,} rows")
        print(f"      └─ Stateful:  {len(df_sf):,} rows")
    for c in FEATS_SL:
        df_sl[c] = pd.to_numeric(df_sl.get(c, 0), errors='coerce').fillna(0)
    aggregation_dict = {feat: 'mean' for feat in FEATS_SL if feat in df_sl.columns}
    if 'label' in df_sl.columns:
        aggregation_dict['label'] = 'first'
    df_sl_agg = df_sl.groupby('tunnel_id').agg(aggregation_dict).reset_index()
    df_sl_agg['tunnel_id'] = df_sl_agg['tunnel_id'].astype(str) + f"_s{seed_name}"
    if verbose:
        print(f"      ├─ Aggregated to {len(df_sl_agg):,} flows")
    for c in FEATS_SF:
        df_sf[c] = pd.to_numeric(df_sf.get(c, 0), errors='coerce').fillna(0)
    df_sf['tunnel_id'] = df_sf['tunnel_id'].astype(str) + f"_s{seed_name}"
    df_merged = pd.merge(df_sl_agg, df_sf, on='tunnel_id', how='inner', suffixes=('', '_sf'))
    if verbose:
        print(f"      └─ Merged: {len(df_merged):,} flows")
    df_merged = df_merged.reindex(columns=ALL_FEATS + ['label', 'tunnel_id'], fill_value=0)
    df_merged['label'] = df_merged['label'].fillna(1).astype(int) if 'label' in df_merged.columns else 1
    df_merged[ALL_FEATS] = df_merged[ALL_FEATS].replace([np.inf, -np.inf], np.nan).fillna(0)
    df_merged['seed'] = int(seed_name)
    return df_merged

def load_aragat_dynamic_seeds(specific_seeds=None, verbose=True):
    if verbose:
        print("\n" + "="*70)
        print("💀 LOADING ARAGAT/MUTANT-DNS DATASET (DYNAMIC SEEDS)")
        print("="*70)
    available_seeds = detect_available_seeds(ARAGAT_DIR, verbose=verbose)
    if not available_seeds:
        raise ValueError("❌ No seeds found in directory!")
    seeds_to_load = specific_seeds if specific_seeds is not None else available_seeds
    if specific_seeds is not None:
        missing = set(seeds_to_load) - set(available_seeds)
        if missing:
            raise ValueError(f"❌ Seeds not found: {sorted(missing)}")
        if verbose:
            print(f"\n📥 Loading {len(seeds_to_load)} seed(s): {', '.join(map(str, seeds_to_load))}")
    elif verbose:
        print(f"\n📥 Loading ALL {len(seeds_to_load)} seed(s): {', '.join(map(str, seeds_to_load))}")
    dfs = []
    for seed in seeds_to_load:
        df_seed = process_single_seed(
            os.path.join(ARAGAT_DIR, f"stateless_features-bridge.pcap_{seed}.csv"),
            os.path.join(ARAGAT_DIR, f"stateful_features-bridge.pcap_{seed}.csv"),
            seed_name=str(seed), verbose=verbose
        )
        dfs.append(df_seed)
    if verbose:
        print(f"\n   🔗 Combining {len(dfs)} seed(s)...")
    df_combined = pd.concat(dfs, ignore_index=True)
    if verbose:
        n_attack = (df_combined['label'] == 1).sum()
        n_benign = (df_combined['label'] == 0).sum()
        print(f"      ✅ Total combined: {len(df_combined):,} flows")
        print(f"         ├─ Attack:  {n_attack:,} ({n_attack/len(df_combined)*100:.1f}%)")
        print(f"         └─ Benign:  {n_benign:,} ({n_benign/len(df_combined)*100:.1f}%)")
    return (df_combined[ALL_FEATS].values,
            df_combined['label'].values,
            df_combined['tunnel_id'].values,
            df_combined['seed'].values)

# =============================================================================
# FIX 1 — extract_tree_importance CON STD REAL PARA RF
# =============================================================================

def extract_tree_importance(model, model_name, feature_names):
    """
    Extrae feature importance de modelos tree-based.
    For RandomForest computes real std among individual trees.
    XGBoost y LightGBM no exponen estimators_ → std = 0.
    """
    importance = model.feature_importances_

    # FIX 1: std among individual trees (only RandomForest has estimators_)
    if hasattr(model, 'estimators_') and model_name == 'RandomForest':
        importances_per_tree = np.array([t.feature_importances_
                                         for t in model.estimators_])
        std = importances_per_tree.std(axis=0)
    else:
        std = np.zeros(len(importance))  # XGB/LGB: no per-tree access

    return pd.DataFrame({
        'feature'   : feature_names,
        'importance': importance,
        'std'       : std,          # ← nuevo
        'model'     : model_name,
    }).sort_values('importance', ascending=False).reset_index(drop=True)

# =============================================================================
# EXTRAER COEFICIENTES DE LOGISTIC REGRESSION
# =============================================================================

def extract_logreg_coefficients(model, feature_names):
    coef = model.coef_[0]
    return pd.DataFrame({
        'feature'        : feature_names,
        'coefficient'    : coef,
        'abs_coefficient': np.abs(coef),
        'std'            : np.zeros(len(coef)),   # sin std para LogReg
        'model'          : 'LogisticRegression',
    }).sort_values('abs_coefficient', ascending=False).reset_index(drop=True)

# =============================================================================
# PERMUTATION IMPORTANCE PARA DL MODELS (CNN Y LSTM)
# =============================================================================

def compute_dl_permutation_importance(model, X_test, y_test, feature_names,
                                      model_name, n_repeats=5):
    print(f"   🔍 Computing {model_name} permutation importance...")
    from sklearn.metrics import f1_score

    class DLWrapper(BaseEstimator):
        def __init__(self, dl_model): self.model = dl_model
        def fit(self, X, y): return self
        def predict(self, X):
            Xr = X.reshape(X.shape[0], X.shape[1], 1) if X.ndim == 2 else X
            return (self.model.predict(Xr, verbose=0) > 0.5).astype(int).flatten()
        def score(self, X, y):
            return f1_score(y, self.predict(X), zero_division=0)

    wrapper = DLWrapper(model)
    result = permutation_importance(
        wrapper, X_test, y_test,
        n_repeats=n_repeats, random_state=42,
        scoring=lambda e, X, y: f1_score(y, e.predict(X), zero_division=0)
    )
    return pd.DataFrame({
        'feature'   : feature_names,
        'importance': result.importances_mean,
        'std'       : result.importances_std,   # already existed
        'model'     : model_name,
    }).sort_values('importance', ascending=False).reset_index(drop=True)

# =============================================================================
# FIX 2 — CSV CONSOLIDADO: 1 fila por feature × columnas por modelo
# =============================================================================

def build_consolidated_csv(importance_dict, recall_dict, save_path=None):
    """
    Genera DataFrame con 23 filas (una por feature) y columnas:
        feature | category |
        rf_importance | rf_std |
        xgb_importance | xgb_std |
        lgb_importance | lgb_std |
        cnn_importance | cnn_std |
        lstm_importance | lstm_std |
        logreg_abs_coef | logreg_std

    FIX 3: columna 'category' con Positional / Stateful / Other
    """
    df_out = pd.DataFrame({'feature': ALL_FEATS})
    df_out['category'] = df_out['feature'].apply(assign_category)   # FIX 3

    model_col_map = {
        'RandomForest'      : ('rf_importance',    'rf_std'),
        'XGBoost'           : ('xgb_importance',   'xgb_std'),
        'LightGBM'          : ('lgb_importance',   'lgb_std'),
        'CNN'               : ('cnn_importance',   'cnn_std'),
        'LSTM'              : ('lstm_importance',  'lstm_std'),
        'LogisticRegression': ('logreg_abs_coef',  'logreg_std'),
    }

    for model_name, (col_imp, col_std) in model_col_map.items():
        if model_name not in importance_dict:
            df_out[col_imp] = np.nan
            df_out[col_std] = np.nan
            continue
        df_m = importance_dict[model_name].copy()
        # LogReg uses 'abs_coefficient'; others use 'importance'
        imp_col = 'abs_coefficient' if model_name == 'LogisticRegression' else 'importance'
        merge_cols = ['feature', imp_col, 'std']
        df_out = df_out.merge(df_m[merge_cols], on='feature', how='left')
        df_out = df_out.rename(columns={imp_col: col_imp, 'std': col_std})

    if save_path:
        df_out.to_csv(save_path, index=False)
        print(f"   ✅ CSV consolidado guardado: {save_path}")
    return df_out

# =============================================================================
# FIX 5 — FIGURA CON BARRAS DE ERROR (xerr=std)
# =============================================================================

def plot_feature_importance_comparison(importance_dict, recall_dict, save_path=None):
    """
    Compara feature importance entre 6 modelos.
    FIX 5: adds xerr=std for RF, CNN, LSTM.
    Colores: rojo = Positional, verde = Stateful, gris = Other.
    """
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Feature Importance (±1 SD): All Models — Positional vs. Stateful',
                 fontsize=15, fontweight='bold')

    COLOR_CAT = {'Positional': '#c0392b', 'Stateful': '#27ae60', 'Other': '#7f8c8d'}

    panels = [
        ('RandomForest',      'importance',     'rf_imp',   axes[0, 0]),
        ('XGBoost',           'importance',     'xgb_imp',  axes[0, 1]),
        ('LightGBM',          'importance',     'lgb_imp',  axes[0, 2]),
        ('LSTM',              'importance',     'lstm_imp', axes[1, 0]),
        ('CNN',               'importance',     'cnn_imp',  axes[1, 1]),
        ('LogisticRegression','abs_coefficient','lr_imp',   axes[1, 2]),
    ]

    for model_name, imp_col, _, ax in panels:
        if model_name not in importance_dict:
            ax.set_visible(False)
            continue
        df_m  = importance_dict[model_name].head(15).copy()
        cats  = df_m['feature'].apply(assign_category)
        colors = [COLOR_CAT[c] for c in cats]
        xerr  = df_m['std'].values if (df_m['std'] > 0).any() else None

        ax.barh(df_m['feature'], df_m[imp_col],
                xerr=xerr,
                color=colors, alpha=0.75,
                error_kw=dict(ecolor='#2c3e50', capsize=3, lw=0.8))
        ax.invert_yaxis()

        recall_pct = recall_dict.get(model_name, 0.0) * 100
        imp_label  = '|Coefficient|' if model_name == 'LogisticRegression' else 'Importance'
        ax.set_xlabel(imp_label, fontsize=9, fontweight='bold')
        ax.set_title(f'{model_name}\n({recall_pct:.2f}% Recall)',
                     fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=8)

    # Leyenda global
    legend_patches = [
        mpatches.Patch(facecolor=COLOR_CAT['Positional'],
                       label='Positional — DNS name patterns'),
        mpatches.Patch(facecolor=COLOR_CAT['Stateful'],
                       label='Stateful — flow statistics'),
        mpatches.Patch(facecolor=COLOR_CAT['Other'], label='Other'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', ncol=3,
               fontsize=10, framealpha=0.85, bbox_to_anchor=(0.5, 0.00))
    plt.tight_layout(rect=[0, 0.04, 1, 0.97])

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"   ✅ Figura guardada: {save_path}")
    plt.show()

# =============================================================================
# TOP FEATURES POR TIPO DE MODELO (sin cambios)
# =============================================================================

def plot_top_features_by_model_type(importance_dict, recall_dict, save_path=None):
    from collections import Counter
    fig, axes = plt.subplots(1, 3, figsize=(22, 6))
    positional_features = POSITIONAL_FEATS

    panels_cfg = [
        (['RandomForest', 'XGBoost', 'LightGBM'], 'Tree-Based Models', 'red',    axes[0]),
        (['LSTM'],                                 'LSTM Model',        'orange', axes[1]),
        (['LogisticRegression', 'CNN'],            'Robust Models',     'green',  axes[2]),
    ]
    for models, title, color, ax in panels_cfg:
        all_feats = []
        for m in models:
            if m in importance_dict:
                imp_col = 'abs_coefficient' if m == 'LogisticRegression' else 'feature'
                all_feats.extend(importance_dict[m].head(5)['feature'].tolist())
        counts = Counter(all_feats)
        df_c = pd.DataFrame(counts.items(), columns=['feature', 'count']
                            ).sort_values('count', ascending=False).head(10)
        bar_colors = ['#c0392b' if f in positional_features else '#27ae60'
                      for f in df_c['feature']]
        ax.barh(df_c['feature'], df_c['count'], color=bar_colors, alpha=0.7)
        avg_recall = np.mean([recall_dict.get(m, 0) for m in models]) * 100
        ax.set_title(f'{title}\n({avg_recall:.2f}% Recall)',
                     fontsize=12, fontweight='bold', color=color)
        ax.set_xlabel('Frequency in Top 5', fontweight='bold')
        ax.invert_yaxis()

    plt.suptitle('Feature Patterns: Vulnerable vs Robust Models',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"   ✅ Saved: {save_path}")
    plt.show()

# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main(specific_seeds=None, subset_size=1500, n_repeats=5):
    print("\n" + "="*70)
    print("🔬 FEATURE IMPORTANCE ANALYSIS V2 — All 6 Models")
    print("="*70)

    X, y, tunnel_ids, seed_labels = load_aragat_dynamic_seeds(
        specific_seeds=specific_seeds, verbose=True)
    print(f"\n✅ Data loaded: {X.shape[0]:,} flows, {X.shape[1]} features")

    scaler_path = os.path.join(MODEL_DIR, "scaler_sota.joblib")
    scaler    = joblib.load(scaler_path)
    X_scaled  = scaler.transform(X)

    importance_dict  = {}
    recall_dict      = {}

    # ── 1. TREE-BASED MODELS ─────────────────────────────────────────────────
    print("\n📊 Tree-based models...")
    for model_name in ['RandomForest', 'XGBoost', 'LightGBM']:
        model_path = os.path.join(MODEL_DIR, f"{model_name}_sota.joblib")
        if not os.path.exists(model_path):
            print(f"   ⚠️  {model_name} not found, skipping.")
            continue
        model = joblib.load(model_path)
        importance_dict[model_name] = extract_tree_importance(model, model_name, ALL_FEATS)
        preds = model.predict(X)                                   # tree models use raw X
        # FIX 4: recall real de la clase ataque
        recall_dict[model_name] = recall_score(y, preds, zero_division=0)
        top5 = importance_dict[model_name].head(5)
        print(f"\n   {model_name} top 5 (recall={recall_dict[model_name]:.3f}):")
        for _, row in top5.iterrows():
            print(f"      {row['feature']:20s}: {row['importance']:.4f} ± {row['std']:.4f}")

    # ── 2. LOGISTIC REGRESSION ───────────────────────────────────────────────
    print("\n📊 LogisticRegression...")
    lr_path = os.path.join(MODEL_DIR, "LogisticRegression_sota.joblib")
    if os.path.exists(lr_path):
        lr_model = joblib.load(lr_path)
        importance_dict['LogisticRegression'] = extract_logreg_coefficients(lr_model, ALL_FEATS)
        preds_lr = lr_model.predict(X_scaled)
        recall_dict['LogisticRegression'] = recall_score(y, preds_lr, zero_division=0)  # FIX 4
        top5 = importance_dict['LogisticRegression'].head(5)
        print(f"   recall={recall_dict['LogisticRegression']:.3f}")
        for _, row in top5.iterrows():
            print(f"      {row['feature']:20s}: {row['coefficient']:+.4f}")
    else:
        print("   ⚠️  LogisticRegression not found, skipping.")

    # ── 3. CNN ────────────────────────────────────────────────────────────────
    print("\n📊 CNN permutation importance...")
    cnn_path = os.path.join(MODEL_DIR, "CNN_sota.keras")
    X_cnn = None
    if os.path.exists(cnn_path):
        cnn_model = keras.models.load_model(cnn_path)
        np.random.seed(42)
        subset_idx = np.random.choice(len(X), size=min(subset_size, len(X)), replace=False)
        X_sub = X_scaled[subset_idx]; y_sub = y[subset_idx]
        importance_dict['CNN'] = compute_dl_permutation_importance(
            cnn_model, X_sub, y_sub, ALL_FEATS, 'CNN', n_repeats=n_repeats)
        X_cnn = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)
        preds_cnn = (cnn_model.predict(X_cnn, verbose=0) > 0.5).astype(int).flatten()
        recall_dict['CNN'] = recall_score(y, preds_cnn, zero_division=0)  # FIX 4
        top5 = importance_dict['CNN'].head(5)
        print(f"   recall={recall_dict['CNN']:.3f}")
        for _, row in top5.iterrows():
            print(f"      {row['feature']:20s}: {row['importance']:.4f} ± {row['std']:.4f}")
    else:
        print("   ⚠️  CNN not found, skipping.")

    # ── 4. LSTM ───────────────────────────────────────────────────────────────
    print("\n📊 LSTM permutation importance...")
    lstm_path = os.path.join(MODEL_DIR, "LSTM_sota.keras")
    if os.path.exists(lstm_path):
        lstm_model = keras.models.load_model(lstm_path)
        if X_cnn is None:   # por si CNN no estaba disponible
            X_cnn = X_scaled.reshape(X_scaled.shape[0], X_scaled.shape[1], 1)
        importance_dict['LSTM'] = compute_dl_permutation_importance(
            lstm_model, X_sub, y_sub, ALL_FEATS, 'LSTM', n_repeats=n_repeats)
        preds_lstm = (lstm_model.predict(X_cnn, verbose=0) > 0.5).astype(int).flatten()
        recall_dict['LSTM'] = recall_score(y, preds_lstm, zero_division=0)  # FIX 4
        top5 = importance_dict['LSTM'].head(5)
        print(f"   recall={recall_dict['LSTM']:.3f}")
        for _, row in top5.iterrows():
            print(f"      {row['feature']:20s}: {row['importance']:.4f} ± {row['std']:.4f}")
    else:
        print("   ⚠️  LSTM not found, skipping.")

    # ── 5. VISUALIZACIONES ────────────────────────────────────────────────────
    print("\n📈 Generando figuras...")
    plot_feature_importance_comparison(
        importance_dict, recall_dict,
        save_path=os.path.join(MODEL_DIR, "feature_importance_with_std.png"))
    plot_top_features_by_model_type(
        importance_dict, recall_dict,
        save_path=os.path.join(MODEL_DIR, "top_features_by_model_type_v2.png"))

    # ── 6. CSV CONSOLIDADO (FIX 2 + FIX 3) ───────────────────────────────────
    print("\n" + "="*70)
    print("📋 CSV CONSOLIDADO (feature × modelo)")
    print("="*70)
    csv_cons = os.path.join(MODEL_DIR, "feature_importance_consolidated.csv")
    df_cons  = build_consolidated_csv(importance_dict, recall_dict, save_path=csv_cons)
    print("\nVista previa (top 6 por rf_importance):")
    preview_cols = [c for c in ['feature','category','rf_importance','rf_std',
                                'cnn_importance','cnn_std','lstm_importance','lstm_std']
                    if c in df_cons.columns]
    print(df_cons.sort_values('rf_importance', ascending=False,
                              na_position='last').head(6)[preview_cols].to_string(index=False))

    # ── 7. SUMMARY BY FEATURE CATEGORY ──────────────────────────────────────
    print("\n" + "="*70)
    print("📊 MEAN IMPORTANCE BY FEATURE CATEGORY")
    print("="*70)
    for col, label in [('rf_importance', 'RF'),
                       ('cnn_importance', 'CNN'),
                       ('lstm_importance', 'LSTM')]:
        if col not in df_cons.columns:
            continue
        grp = df_cons.groupby('category')[col].mean().sort_values(ascending=False)
        print(f"\n{label}:")
        for cat, val in grp.items():
            print(f"   {cat:12s}: {val:.4f}")

    # ── 8. SUMMARY TABLE (igual que original) ─────────────────────────────────
    print("\n" + "="*70)
    print("📊 QUANTITATIVE SUMMARY")
    print("="*70)
    summary = []
    for model_name, df_m in importance_dict.items():
        top10 = df_m.head(10)['feature'].tolist()
        n_pos = sum(f in POSITIONAL_FEATS for f in top10)
        n_sta = sum(f in STATEFUL_FEATS   for f in top10)
        top1  = df_m.iloc[0]['feature']
        summary.append({
            'model'              : model_name,
            'top1_feature'       : top1,
            'category_top1'      : assign_category(top1),
            'positional_in_top10': n_pos,
            'stateful_in_top10'  : n_sta,
            'recall'             : f"{recall_dict.get(model_name, 0):.4f}",
        })
    df_summary = pd.DataFrame(summary).sort_values('recall')
    print("\n" + df_summary.to_string(index=False))
    df_summary.to_csv(os.path.join(MODEL_DIR, "feature_analysis_summary_v2.csv"), index=False)

    print("\n📁 Archivos guardados:")
    print(f"   ├─ feature_importance_with_std.png")
    print(f"   ├─ top_features_by_model_type_v2.png")
    print(f"   ├─ feature_importance_consolidated.csv   ← NUEVO")
    print(f"   └─ feature_analysis_summary_v2.csv")
    print("\n" + "="*70)
    print("✅ FEATURE IMPORTANCE V2 COMPLETO")
    print("="*70)

    return importance_dict, recall_dict, df_cons

# =============================================================================
# EXECUTION — COLAB COMPATIBLE
# =============================================================================

if __name__ == "__main__":
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    if IN_COLAB:
        SPECIFIC_SEEDS = None   # None = todos los disponibles
        SUBSET_SIZE    = 1500
        N_REPEATS      = 5
        print("🔬 Google Colab mode")
        importance_dict, recall_dict, df_consolidated = main(
            specific_seeds=SPECIFIC_SEEDS,
            subset_size=SUBSET_SIZE,
            n_repeats=N_REPEATS,
        )
    else:
        import argparse
        parser = argparse.ArgumentParser(description='Feature Importance V2')
        parser.add_argument('--seeds', type=int, nargs='+', default=None)
        parser.add_argument('--subset-size', type=int, default=1500)
        parser.add_argument('--n-repeats', type=int, default=5)
        args = parser.parse_args()
        importance_dict, recall_dict, df_consolidated = main(
            specific_seeds=args.seeds,
            subset_size=args.subset_size,
            n_repeats=args.n_repeats,
        )


In [ ]:
# =============================================================================
# STATISTICAL TESTS + LATEX TABLE
# Dos tareas independientes:
#
# TAREA 1 — Tabla LaTeX (lee feature_importance_consolidated.csv)
#   Top 10 features por RF | Feature | Category | RF±std | CNN±std | LSTM±std | LogReg
#   Separador entre Positional y Stateful
#   Importancias negativas se reportan tal cual (son hallazgo, no error)
#
# TASK 2 — Statistical tests (paired t-test + Cohen's d)
#   Para cada modelo: recall en los 9 seeds individuales
#   Comparaciones: cada modelo robusto vs cada modelo fallido
#   Resultado: tabla LaTeX de p-values y Cohen's d
#
# USO EN COLAB:
#   Subir este script y ejecutar como nueva celda.
#   Does not require feature_importance_v2.py to be in memory.
# =============================================================================

import os
import re
import numpy as np
import pandas as pd
import joblib
from scipy.stats import ttest_rel, ttest_ind
from sklearn.metrics import recall_score
from tensorflow import keras

# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR   = "/content/drive/MyDrive/Tunnel/CSV_CIC21"
MODEL_DIR  = os.path.join(BASE_DIR, "Models_SOTA_Hybrid")
ARAGAT_DIR = os.path.join(BASE_DIR, "CSV_Generated")
CSV_CONS   = os.path.join(MODEL_DIR, "feature_importance_consolidated.csv")

FEATS_SL = [
    "FQDN_count", "subdomain_length", "upper", "lower", "numeric",
    "entropy", "special", "labels", "labels_max", "labels_average", "len"
]
FEATS_SF = [
    "rr", "A_frequency", "AAAA_frequency", "CNAME_frequency", "TXT_frequency",
    "MX_frequency", "NS_frequency", "NULL_frequency",
    "rr_count", "distinct_ip", "unique_ttl", "total_queries"
]
ALL_FEATS = FEATS_SL + FEATS_SF


# =============================================================================
# TAREA 1 — TABLA LaTeX DESDE CSV CONSOLIDADO
# =============================================================================

def fmt_imp(importance, std, decimals=4):
    """
    Formatea importance ± std para LaTeX.
    Negativo → reportar tal cual con signo (hallazgo, no error).
    """
    if pd.isna(importance):
        return r"\textemdash"
    sign = "-" if importance < 0 else ""
    imp_str = f"{abs(importance):.{decimals}f}"
    std_str = f"{abs(std):.{decimals}f}" if not pd.isna(std) and std > 0 else "---"
    if std_str == "---":
        return f"{sign}{imp_str}"
    return f"${sign}{imp_str}{{\pm}}{std_str}$"


def generate_latex_table(csv_path, top_n=10, save_path=None):
    """
    Genera tabla LaTeX con top-N features.
    Columnas: Feature | Category | RF (mean±std) | CNN | LSTM | LogReg |coef|
    Separador visual entre Positional y Stateful.
    """
    df = pd.read_csv(csv_path)

    # Top 10 por RF importance
    df_top = df.nlargest(top_n, 'rf_importance').copy()
    df_top = df_top.sort_values('rf_importance', ascending=False).reset_index(drop=True)

    # ── Construir filas ───────────────────────────────────────────────────────
    lines = []

    # Header
    lines.append(r"\begin{table}[t]")
    lines.append(r"  \centering")
    lines.append(r"  \caption{Top-10 features ranked by Random Forest importance.")
    lines.append(r"  Values show mean~$\pm$~std. Negative permutation importance")
    lines.append(r"  (CNN/LSTM) indicates the feature acts as noise for that model.")
    lines.append(r"  \textsuperscript{$\dagger$}Std between individual trees.")
    lines.append(r"  \textsuperscript{$\ddagger$}Std from $n{=}5$ permutation repeats.}")
    lines.append(r"  \label{tab:feature_importance}")
    lines.append(r"  \resizebox{\columnwidth}{!}{%")
    lines.append(r"  \begin{tabular}{llcccc}")
    lines.append(r"  \toprule")
    lines.append(r"  \textbf{Feature} & \textbf{Cat.} & "
                 r"\textbf{RF\textsuperscript{$\dagger$}} & "
                 r"\textbf{CNN\textsuperscript{$\ddagger$}} & "
                 r"\textbf{LSTM\textsuperscript{$\ddagger$}} & "
                 r"\textbf{LogReg $|\beta|$} \\")
    lines.append(r"  \midrule")

    prev_cat = None
    for _, row in df_top.iterrows():
        cat = row['category']

        # Category group separator
        if prev_cat is not None and cat != prev_cat:
            lines.append(r"  \midrule")
        prev_cat = cat

        feat_name = row['feature'].replace('_', r'\_')
        cat_short = 'Pos.' if cat == 'Positional' else ('Sta.' if cat == 'Stateful' else 'Oth.')

        rf_str   = fmt_imp(row.get('rf_importance'),  row.get('rf_std'))
        cnn_str  = fmt_imp(row.get('cnn_importance'), row.get('cnn_std'))
        lstm_str = fmt_imp(row.get('lstm_importance'),row.get('lstm_std'))
        lr_str   = fmt_imp(row.get('logreg_abs_coef'),row.get('logreg_std'), decimals=4)

        lines.append(f"  \\texttt{{{feat_name}}} & {cat_short} & "
                     f"{rf_str} & {cnn_str} & {lstm_str} & {lr_str} \\\\")

    lines.append(r"  \bottomrule")
    lines.append(r"  \end{tabular}}")
    lines.append(r"  \end{table}")

    table_str = "\n".join(lines)

    if save_path:
        with open(save_path, 'w') as f:
            f.write(table_str)
        print(f"   ✅ Tabla LaTeX guardada: {save_path}")

    print("\n" + "="*70)
    print("TABLA LaTeX — copiar al paper:")
    print("="*70)
    print(table_str)
    return table_str


# =============================================================================
# TASK 2 — RECALL PER SEED (individual evaluation)
# =============================================================================

def load_single_seed_data(seed, scaler):
    """Loads and scales data for ONE specific seed."""
    path_sl = os.path.join(ARAGAT_DIR,
                           f"stateless_features-bridge.pcap_{seed}.csv")
    path_sf = os.path.join(ARAGAT_DIR,
                           f"stateful_features-bridge.pcap_{seed}.csv")
    df_sl = pd.read_csv(path_sl)
    df_sf = pd.read_csv(path_sf)

    for c in FEATS_SL:
        df_sl[c] = pd.to_numeric(df_sl.get(c, 0), errors='coerce').fillna(0)
    for c in FEATS_SF:
        df_sf[c] = pd.to_numeric(df_sf.get(c, 0), errors='coerce').fillna(0)

    agg_dict = {f: 'mean' for f in FEATS_SL if f in df_sl.columns}
    if 'label' in df_sl.columns:
        agg_dict['label'] = 'first'
    df_sl_agg = df_sl.groupby('tunnel_id').agg(agg_dict).reset_index()
    df_sl_agg['tunnel_id'] = df_sl_agg['tunnel_id'].astype(str) + f"_s{seed}"
    df_sf['tunnel_id'] = df_sf['tunnel_id'].astype(str) + f"_s{seed}"

    df = pd.merge(df_sl_agg, df_sf, on='tunnel_id', how='inner', suffixes=('', '_sf'))
    df = df.reindex(columns=ALL_FEATS + ['label'], fill_value=0)
    df['label'] = df['label'].fillna(1).astype(int)
    df[ALL_FEATS] = df[ALL_FEATS].replace([np.inf, -np.inf], np.nan).fillna(0)

    X = scaler.transform(df[ALL_FEATS].values)
    y = df['label'].values
    return X, y


def compute_per_seed_recalls(seeds, models_dict, scaler):
    """
    Para cada modelo y cada seed: calcula recall de la clase ataque.

    Returns:
        recalls: dict {model_name: np.array(n_seeds)}
    """
    recalls = {m: [] for m in models_dict}

    for seed in seeds:
        print(f"   Evaluando seed {seed}...", end=' ')
        X_seed, y_seed = load_single_seed_data(seed, scaler)
        X_3d = X_seed.reshape(X_seed.shape[0], X_seed.shape[1], 1)

        for model_name, model in models_dict.items():
            if model is None:
                recalls[model_name].append(np.nan)
                continue
            if model_name in ('CNN', 'LSTM'):
                preds = (model.predict(X_3d, verbose=0) > 0.5).astype(int).flatten()
            else:
                # Tree models usan X sin escalar (mismo scaler que entrenamiento)
                # LogReg usa X escalado
                preds = model.predict(X_seed if model_name != 'LogisticRegression'
                                      else X_seed)
            recalls[model_name].append(recall_score(y_seed, preds, zero_division=0))

        recalled = {m: f"{recalls[m][-1]:.3f}" for m in models_dict if models_dict[m]}
        print(" | ".join(f"{m}={v}" for m, v in recalled.items()))

    return {m: np.array(v) for m, v in recalls.items()}


# =============================================================================
# TASK 2 — STATISTICAL TESTS (paired t-test + Cohen's d)
# =============================================================================

def cohens_d_paired(a, b):
    """Cohen's d para muestras pareadas: d = mean(diff) / std(diff)."""
    diff = np.array(a) - np.array(b)
    return diff.mean() / (diff.std(ddof=1) + 1e-12)


def run_statistical_tests(recalls_dict, save_path=None):
    """
    Comparaciones pareadas entre todos los pares de modelos.
    Test: ttest_rel (paired, more powerful because the same seeds are compared).
    Efecto: Cohen's d pareado.

    Also compares WITHIN each model: recall vs 0 (baseline).
    """
    models    = [m for m, v in recalls_dict.items() if not np.all(np.isnan(v))]
    n_seeds   = len(next(iter(recalls_dict.values())))

    print("\n" + "="*70)
    print(f"STATISTICAL TESTS — {n_seeds} seeds (paired t-test)")
    print("="*70)

    print("\n📊 Recall por seed (media ± std):")
    for m in models:
        v = recalls_dict[m]
        print(f"   {m:22s}: {v.mean():.4f} ± {v.std():.4f}  "
              f"[{v.min():.3f} – {v.max():.3f}]")

    # ── Comparaciones entre pares ────────────────────────────────────────────
    results = []
    print(f"\n📊 Comparaciones pareadas (α=0.05):")
    print(f"   {'Model A':22s}  {'Model B':22s}  {'Δmean':>8}  {'t':>7}  {'p':>8}  {'d':>6}  Sig")
    print("   " + "─"*80)

    for i in range(len(models)):
        for j in range(i + 1, len(models)):
            a_name, b_name = models[i], models[j]
            a = recalls_dict[a_name]
            b = recalls_dict[b_name]

            # Manejo de NaN
            mask = ~(np.isnan(a) | np.isnan(b))
            if mask.sum() < 3:
                continue
            a_c, b_c = a[mask], b[mask]

            delta = a_c.mean() - b_c.mean()

            # Si ambas distribuciones tienen varianza cero (ej. XGB=0 vs LGB=0)
            # el t-test devuelve NaN — detectar antes de llamarlo
            if np.var(a_c - b_c) < 1e-12:
                t_stat, p_val, d, sig = np.nan, np.nan, 0.0, "†"
            else:
                t_stat, p_val = ttest_rel(a_c, b_c)
                d             = cohens_d_paired(a_c, b_c)
                sig           = "***" if p_val < 0.001 else ("**" if p_val < 0.01
                                else ("*" if p_val < 0.05 else "ns"))

            results.append({
                'model_a'  : a_name,
                'model_b'  : b_name,
                'mean_a'   : a_c.mean(),
                'mean_b'   : b_c.mean(),
                'delta'    : delta,
                't_stat'   : t_stat,
                'p_value'  : p_val,
                'cohens_d' : d,
                'sig'      : sig,
                'n'        : int(mask.sum()),
            })
            t_str = f"{t_stat:7.3f}" if not np.isnan(t_stat) else "    ---"
            p_str = f"{p_val:8.4f}" if not np.isnan(p_val) else "     ---"
            print(f"   {a_name:22s}  {b_name:22s}  "
                  f"{delta:+8.4f}  {t_str}  {p_str}  {d:6.2f}  {sig}")

    df_stats = pd.DataFrame(results)

    # ── Tabla LaTeX de tests ──────────────────────────────────────────────────
    latex_stats = _build_stats_latex_table(df_stats, n_seeds)
    print("\n" + "="*70)
    print("LaTeX TABLE — Statistical tests (copy to paper):")
    print("="*70)
    print(latex_stats)

    if save_path:
        df_stats.to_csv(save_path, index=False)
        print(f"\n   ✅ CSV guardado: {save_path}")
        tex_path = save_path.replace('.csv', '.tex')
        with open(tex_path, 'w') as f:
            f.write(latex_stats)
        print(f"   ✅ LaTeX guardado: {tex_path}")

    return df_stats


def _build_stats_latex_table(df_stats, n_seeds):
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"  \centering")
    lines.append(r"  \caption{Pairwise paired $t$-tests on per-seed recall")
    lines.append(r"  ($n=" + str(n_seeds) + r"$ seeds, Mutant Payload).")
    lines.append(r"  $^{*}p{<}0.05$, $^{**}p{<}0.01$, $^{***}p{<}0.001$, ns: not significant.}")
    lines.append(r"  Cohen's~$d$ for paired differences: small $|d|{<}0.5$,")
    lines.append(r"  medium $0.5{\le}|d|{<}0.8$, large $|d|{\ge}0.8$.}")
    lines.append(r"  \label{tab:stat_tests}")
    lines.append(r"  \begin{tabular}{llcccc}")
    lines.append(r"  \toprule")
    lines.append(r"  \textbf{Model A} & \textbf{Model B} & "
                 r"$\overline{r}_A$ & $\overline{r}_B$ & "
                 r"$p$-value & Cohen's~$d$ \\")
    lines.append(r"  \midrule")

    for _, row in df_stats.iterrows():
        sig = row['sig']
        ma  = row['model_a'].replace('_', r'\_').replace('LogisticRegression', 'LogReg')
        mb  = row['model_b'].replace('_', r'\_').replace('LogisticRegression', 'LogReg')

        # NaN p-value: both models have identical recall (zero variance)
        if pd.isna(row['p_value']):
            p_cell = r"\multicolumn{1}{c}{---\textsuperscript{$\dagger$}}"
        elif row['p_value'] < 0.0001:
            p_cell = f"$<0.0001^{{{sig}}}$"
        else:
            p_cell = f"${row['p_value']:.4f}^{{{sig}}}$"

        lines.append(
            f"  {ma} & {mb} & "
            f"{row['mean_a']:.4f} & {row['mean_b']:.4f} & "
            f"{p_cell} & ${row['cohens_d']:+.2f}$ \\\\"
        )

    lines.append(r"  \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


# =============================================================================
# SCENARIO B vs C — Recall drop per model (CIC-2021 → Mutant Payload)
# =============================================================================

def load_cic_attack_data(scaler, n_bootstrap=9, bootstrap_size=1500, random_state=42):
    """
    Carga los flows de ATAQUE del dataset CIC-2021 (carpeta Attacks/).
    Combina stateless (media por tunnel) + stateful (fila).
    Genera n_bootstrap subconjuntos aleatorios para emparejamiento con los seeds.

    Returns:
        X_boots  : list of n_bootstrap arrays (bootstrap_size, 23)
        y_boots  : list of n_bootstrap arrays (todos 1 = ataque)
    """
    import glob as _glob

    print(f"\n   🔍 Buscando archivos CIC attack en {BASE_DIR}...")

    def _is_attack(path):
        return os.path.basename(os.path.dirname(path)).lower() == "attacks"

    sf_all = [f for f in _glob.glob(
        os.path.join(BASE_DIR, "**", "stateful_features*.csv"), recursive=True)
        if _is_attack(f)]
    sl_all = [f for f in _glob.glob(
        os.path.join(BASE_DIR, "**", "stateless_features*.csv"), recursive=True)
        if _is_attack(f)]

    print(f"   CIC attack files — stateful: {len(sf_all)}, stateless: {len(sl_all)}")
    if not sf_all:
        raise FileNotFoundError("No CIC attack stateful files found.")

    # Emparejar por nombre de archivo
    sf_map = {os.path.basename(p).replace("stateful", "PLACEHOLDER"): p for p in sf_all}
    sl_map = {os.path.basename(p).replace("stateless", "PLACEHOLDER"): p for p in sl_all}
    paired = [(sf_map[k], sl_map[k]) for k in sorted(set(sf_map) & set(sl_map))]
    if not paired:
        paired = list(zip(sorted(sf_all), sorted(sl_all)))
        print(f"   ⚠️  Emparejamiento por nombre fallido, usando orden ({len(paired)} pares).")
    else:
        print(f"   ✅ {len(paired)} pares emparejados.")

    rows_X = []
    for sf_path, sl_path in paired:
        try:
            df_sf = pd.read_csv(sf_path)
            df_sl = pd.read_csv(sl_path)

            # FIX: verificar existencia antes de to_numeric (evita 'int'.fillna())
            for c in FEATS_SL:
                if c in df_sl.columns:
                    df_sl[c] = pd.to_numeric(df_sl[c], errors='coerce').fillna(0)
                else:
                    df_sl[c] = 0.0
            for c in FEATS_SF:
                if c in df_sf.columns:
                    df_sf[c] = pd.to_numeric(df_sf[c], errors='coerce').fillna(0)
                else:
                    df_sf[c] = 0.0

            # Calcular total_queries
            freq_cols = [c for c in df_sf.columns if '_frequency' in c]
            if freq_cols:
                df_sf['total_queries'] = df_sf[freq_cols].sum(axis=1)
            elif 'rr_count' in df_sf.columns:
                df_sf['total_queries'] = df_sf['rr_count']
            else:
                df_sf['total_queries'] = 1

            # Inyectar medias SL en cada fila SF
            for feat in FEATS_SL:
                df_sf[feat] = df_sl[feat].mean() if feat in df_sl.columns else 0.0

            rows_X.append(df_sf[ALL_FEATS].fillna(0).values)
        except Exception as e:
            print(f"   ⚠️  Error en {os.path.basename(sf_path)}: {e}")
            continue

    if not rows_X:
        raise ValueError("No CIC rows loaded.")

    X_all = np.vstack(rows_X)
    X_all = scaler.transform(X_all)
    y_all = np.ones(len(X_all), dtype=int)   # todos son ataques
    print(f"   ✅ CIC attack total: {len(X_all):,} flows")

    # Generar n_bootstrap subconjuntos (matching size de seeds de Mutant)
    rng = np.random.default_rng(random_state)
    X_boots, y_boots = [], []
    for i in range(n_bootstrap):
        idx = rng.choice(len(X_all), size=min(bootstrap_size, len(X_all)), replace=False)
        X_boots.append(X_all[idx])
        y_boots.append(y_all[idx])

    return X_boots, y_boots


def compare_scenario_b_vs_c(models_dict, recalls_c, scaler,
                             n_bootstrap=9, bootstrap_size=1500,
                             save_path=None):
    """
    Scenario B: recall en CIC-2021 ataque (bootstrap)
    Scenario C: recall en Mutant Payload (por seed) — ya calculado

    Para cada modelo: paired t-test B vs C + Cohen's d.
    Question: "Is the recall drop when mutating the attack statistically significant?"
    """
    print("\n" + "="*70)
    print("SCENARIO B vs C — Recall drop per model (CIC-2021 → Mutant)")
    print("="*70)

    try:
        X_boots, y_boots = load_cic_attack_data(
            scaler, n_bootstrap=n_bootstrap, bootstrap_size=bootstrap_size)
    except Exception as e:
        print(f"   ❌ No se pudo cargar CIC data: {e}")
        return None

    # Calcular recalls B (CIC bootstrap) para cada modelo
    recalls_b = {m: [] for m in models_dict}
    for i, (X_b, y_b) in enumerate(zip(X_boots, y_boots)):
        X_b3 = X_b.reshape(X_b.shape[0], X_b.shape[1], 1)
        for model_name, model in models_dict.items():
            if model is None:
                recalls_b[model_name].append(np.nan)
                continue
            if model_name in ('CNN', 'LSTM'):
                preds = (model.predict(X_b3, verbose=0) > 0.5).astype(int).flatten()
            else:
                preds = model.predict(X_b)
            recalls_b[model_name].append(recall_score(y_b, preds, zero_division=0))

    recalls_b = {m: np.array(v) for m, v in recalls_b.items()}

    # Tabla comparativa y tests
    results = []
    print(f"\n   {'Model':22s}  {'B (CIC)':>10}  {'C (Mutant)':>12}  {'Δ':>8}  {'t':>7}  {'p':>8}  {'d':>6}  Sig")
    print("   " + "─"*85)

    for model_name in models_dict:
        b = recalls_b[model_name]
        c = np.array(recalls_c.get(model_name, [np.nan] * n_bootstrap))
        mask = ~(np.isnan(b) | np.isnan(c))
        if mask.sum() < 3:
            continue
        b_c, c_c = b[mask], c[mask]
        delta = b_c.mean() - c_c.mean()   # positive = B > C (expected drop)

        if np.var(b_c - c_c) < 1e-12:
            t_stat, p_val, d, sig = np.nan, np.nan, 0.0, "†"
        else:
            t_stat, p_val = ttest_rel(b_c, c_c)
            d   = cohens_d_paired(b_c, c_c)
            sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01
                  else ("*" if p_val < 0.05 else "ns"))

        results.append({'model': model_name,
                        'recall_B': b_c.mean(), 'recall_C': c_c.mean(),
                        'delta': delta, 't_stat': t_stat, 'p_value': p_val,
                        'cohens_d': d, 'sig': sig})
        t_str = f"{t_stat:7.3f}" if not np.isnan(t_stat) else "    ---"
        p_str = f"{p_val:8.4f}" if not np.isnan(p_val) else "     ---"
        print(f"   {model_name:22s}  {b_c.mean():10.4f}  {c_c.mean():12.4f}  "
              f"{delta:+8.4f}  {t_str}  {p_str}  {d:6.2f}  {sig}")

    df_bc = pd.DataFrame(results)

    # Tabla LaTeX
    latex_bc = _build_bc_latex_table(df_bc)
    print("\n" + "="*70)
    print("TABLA LaTeX — Scenario B vs C (copiar al paper):")
    print("="*70)
    print(latex_bc)

    if save_path:
        df_bc.to_csv(save_path, index=False)
        tex_path = save_path.replace('.csv', '.tex')
        with open(tex_path, 'w') as f:
            f.write(latex_bc)
        print(f"\n   ✅ Guardado: {save_path}  +  {tex_path}")

    return df_bc


def _build_bc_latex_table(df_bc):
    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"  \centering")
    lines.append(r"  \caption{Statistical significance of recall drop from")
    lines.append(r"  Scenario~B (standard CIC-2021 attack, bootstrap $n{=}9$)")
    lines.append(r"  to Scenario~C (Mutant Payload, $n{=}9$ seeds).")
    lines.append(r"  Paired $t$-test; $\dagger$indeterminate (zero variance).")
    lines.append(r"  $^{***}p{<}0.001$.}")
    lines.append(r"  \label{tab:scenario_bc}")
    lines.append(r"  \begin{tabular}{lcccccc}")
    lines.append(r"  \toprule")
    lines.append(r"  \textbf{Model} & $\overline{r}_B$ & $\overline{r}_C$ & "
                 r"$\Delta r$ & $t$ & $p$-value & $d$ \\")
    lines.append(r"  \midrule")
    for _, row in df_bc.iterrows():
        mname = row['model'].replace('LogisticRegression', 'LogReg')
        if pd.isna(row['p_value']):
            p_cell = r"---\textsuperscript{$\dagger$}"
            t_cell = "---"
        else:
            p_cell = (f"$<0.0001^{{{row['sig']}}}$"
                      if row['p_value'] < 0.0001
                      else f"${row['p_value']:.4f}^{{{row['sig']}}}$")
            t_cell = f"${row['t_stat']:.2f}$"
        lines.append(
            f"  {mname} & {row['recall_B']:.4f} & {row['recall_C']:.4f} & "
            f"${row['delta']:+.4f}$ & {t_cell} & {p_cell} & ${row['cohens_d']:+.2f}$ \\\\"
        )
    lines.append(r"  \bottomrule")
    lines.append(r"  \end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


# =============================================================================
# PUNTO DE ENTRADA PRINCIPAL
# =============================================================================

def run_all(seeds=None):
    print("\n" + "="*70)
    print("STATISTICAL ANALYSIS + LaTeX TABLES")
    print("="*70)

    # ── TAREA 1: Tabla de feature importance ──────────────────────────────────
    if not os.path.exists(CSV_CONS):
        print(f"⚠️  CSV consolidado no encontrado: {CSV_CONS}")
        print("   Ejecuta feature_importance_v2.py primero.")
    else:
        generate_latex_table(
            CSV_CONS,
            top_n=10,
            save_path=os.path.join(MODEL_DIR, "table_feature_importance.tex")
        )

    # ── TASK 2: Statistical tests ────────────────────────────────────────────
    print("\n" + "="*70)
    print("CARGANDO MODELOS PARA TESTS POR SEED...")
    print("="*70)

    scaler = joblib.load(os.path.join(MODEL_DIR, "scaler_sota.joblib"))

    # Auto-detect available seeds
    if seeds is None:
        pattern = re.compile(r'stateless_features-bridge\.pcap_(\d+)\.csv')
        avail_files = os.listdir(ARAGAT_DIR)
        seeds = sorted({int(m.group(1))
                        for f in avail_files
                        if (m := pattern.match(f)) and
                        os.path.exists(os.path.join(
                            ARAGAT_DIR,
                            f"stateful_features-bridge.pcap_{m.group(1)}.csv"))})
    print(f"   Seeds: {seeds}")

    # Cargar modelos
    models_dict = {}
    model_files = {
        'RandomForest'      : 'RandomForest_sota.joblib',
        'XGBoost'           : 'XGBoost_sota.joblib',
        'LightGBM'          : 'LightGBM_sota.joblib',
        'LogisticRegression': 'LogisticRegression_sota.joblib',
        'CNN'               : 'CNN_sota.keras',
        'LSTM'              : 'LSTM_sota.keras',
    }
    for name, fname in model_files.items():
        fpath = os.path.join(MODEL_DIR, fname)
        if not os.path.exists(fpath):
            print(f"   ⚠️  {name} no encontrado, omitido.")
            models_dict[name] = None
            continue
        if fname.endswith('.keras'):
            models_dict[name] = keras.models.load_model(fpath)
        else:
            models_dict[name] = joblib.load(fpath)
        print(f"   ✅ {name} cargado.")

    # Recall por seed
    print(f"\n📊 Calculando recall por seed ({len(seeds)} seeds × {len(models_dict)} modelos)...")
    recalls = compute_per_seed_recalls(seeds, models_dict, scaler)

    # Save recalls to CSV (useful for manual inspection)
    df_recalls = pd.DataFrame(recalls, index=[f"seed_{s}" for s in seeds])
    recalls_csv = os.path.join(MODEL_DIR, "recalls_per_seed.csv")
    df_recalls.to_csv(recalls_csv)
    print(f"\n   ✅ Recalls por seed guardados: {recalls_csv}")
    print("\n" + df_recalls.to_string())

    # Statistical tests model vs model
    df_stats = run_statistical_tests(
        recalls,
        save_path=os.path.join(MODEL_DIR, "statistical_tests.csv")
    )

    # ── SCENARIO B vs C: recall drop CIC → Mutant per model ──────────────────
    df_bc = compare_scenario_b_vs_c(
        models_dict    = models_dict,
        recalls_c      = recalls,           # Scenario C = lo que ya calculamos
        scaler         = scaler,
        n_bootstrap    = len(seeds),        # mismo n que seeds de Mutant
        bootstrap_size = 1500,
        save_path      = os.path.join(MODEL_DIR, "scenario_b_vs_c.csv")
    )

    return df_recalls, df_stats, df_bc


# =============================================================================
# COLAB EXECUTION
# =============================================================================

if __name__ == "__main__":
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    df_recalls, df_stats, df_bc = run_all()


## Figure 2: CNN Normalized Mean Activation Profiles

Generate Figure 2 from the paper: normalized mean activation profiles for each feature group
across the first Conv1D layer of the baseline CNN model.

In [ ]:
# =============================================================================
# FIGURA 2: NORMALIZED MEAN ACTIVATION PROFILES - CNN FEATURE GROUP ACTIVATION
# FINAL version - Fixes: panel (a) annotation, dual shading, exact caption
# =============================================================================
# Script autocontenido: no requiere celdas previas del notebook.
# Solo necesita BASE_DIR y MODEL_DIR_SOTA definidos abajo.
# =============================================================================

import os
import glob
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# =============================================================================
# DEFINICIONES DE FEATURES (copiadas del notebook Model_CIC(3).ipynb)
# =============================================================================

FEATS_SL = [
    "FQDN_count", "subdomain_length", "upper", "lower", "numeric",
    "entropy", "special", "labels", "labels_max", "labels_average", "len"
]

FEATS_SF = [
    "rr",
    "A_frequency", "AAAA_frequency", "CNAME_frequency", "TXT_frequency",
    "MX_frequency", "NS_frequency", "NULL_frequency",
    "rr_count", "distinct_ip", "unique_ttl", "total_queries"
]

ALL_FEATS = FEATS_SL + FEATS_SF   # 23 features total


def clean_and_prepare(df_sl, df_sf, row_idx, scaler):
    """Combina stateless (mean) + stateful (row) y genera input (1, 23, 1) para la CNN."""
    sf_row = df_sf.iloc[row_idx:row_idx + 1].copy()

    # Deserializar columnas que pueden venir como string-set (e.g. "{'1.2.3.4'}")
    for col in ['distinct_ip', 'unique_ttl']:
        if col in sf_row.columns:
            val = sf_row[col].iloc[0]
            try:
                if isinstance(val, str):
                    sf_row[col] = len(eval(val)) if val != 'set()' else 0
                else:
                    sf_row[col] = float(val)
            except Exception:
                sf_row[col] = 0

    # Calcular total_queries sumando todas las columnas *_frequency
    freq_cols = [c for c in sf_row.columns if '_frequency' in c]
    if freq_cols:
        sf_row['total_queries'] = sf_row[freq_cols].sum(axis=1)
    elif 'rr_count' in sf_row.columns:
        sf_row['total_queries'] = sf_row['rr_count']
    else:
        sf_row['total_queries'] = 1

    # Inyectar medias de features stateless en la fila stateful
    for feat in FEATS_SL:
        if feat in df_sl.columns:
            sf_row[feat] = df_sl[feat].mean()
        else:
            sf_row[feat] = 0.0

    # Seleccionar las 23 features y escalar
    x = scaler.transform(sf_row[ALL_FEATS].fillna(0))
    return x.reshape(1, 23, 1)

# =============================================================================
# PASO 0: PRE-VERIFICATION
# =============================================================================

print("=" * 60)
print("STEP 0: MODEL AND DATA VERIFICATION")
print("=" * 60)

# --- Verificar existencia del modelo ---
cnn_path = os.path.join(MODEL_DIR_SOTA, "CNN_sota.keras")
scaler_path = os.path.join(MODEL_DIR_SOTA, "scaler_sota.joblib")

if not os.path.exists(cnn_path):
    raise FileNotFoundError(
        f"\n❌ CNN_sota.keras no encontrado en:\n   {cnn_path}\n"
        f"   Verify that the model is in Models_SOTA_Hybrid/"
    )
if not os.path.exists(scaler_path):
    raise FileNotFoundError(
        f"\n❌ scaler_sota.joblib no encontrado en:\n   {scaler_path}"
    )

print(f"✅ Modelo CNN encontrado: {cnn_path}")
print(f"✅ Scaler encontrado:     {scaler_path}")

# --- Cargar modelo y scaler ---
cnn_model = load_model(cnn_path)
scaler    = joblib.load(scaler_path)
print("\n📐 Arquitectura CNN:")
cnn_model.summary()

# --- Identify last Conv1D layer and its dimensions ---
conv_layers = [l for l in cnn_model.layers
               if isinstance(l, tf.keras.layers.Conv1D)]
if not conv_layers:
    raise ValueError("❌ El modelo no tiene capas Conv1D.")

# Using the FIRST Conv1D (output: 23 × 64) instead of the last (5 × 256).
# Ventaja: con padding='same' la primera capa tiene D=23 salidas, una por feature.
# This gives 1:1 mapping between spatial position and input feature, making
# el eje X directamente interpretable (pos 0 = FQDN_count, ..., pos 22 = total_queries).
first_conv = conv_layers[0]
last_conv  = conv_layers[-1]   # conservado para referencia en el reporte

act_model = Model(inputs=cnn_model.input, outputs=first_conv.output)
D = act_model.output_shape[1]   # dimensiones espaciales (= 23 con primera Conv1D)
F = act_model.output_shape[2]   # number of filters

print(f"\n📊 Conv1D layer used for activation: '{first_conv.name}' (1st layer, D=23)")
print(f"   Output shape: (None, {D}, {F})  →  D={D} puntos espaciales, F={F} filtros")

if D < 4:
    print(f"\n⚠️  WARNING: Only {D} spatial points in the last Conv1D.")
    print("   The figure will have fewer points but the conceptual message remains valid.")
    print("   If more resolution is needed, consider using an intermediate Conv1D layer.")
else:
    print(f"\n✅ D={D} puntos espaciales — suficiente para una figura informativa.")

# --- Verificar archivos disponibles ---
# CIC 2021: la etiqueta viene del directorio padre.
# Estructura: .../Attacks/stateful_features-*.csv   ← ataque
#             .../Benign/stateful_features-*.csv    ← benigno
# Pre-filtering here to pass ONLY attack files to collect_attack_flows.

def _is_cic_attack(path):
    """True si el folder inmediato padre es 'Attacks' (case-insensitive)."""
    return os.path.basename(os.path.dirname(path)).lower() == "attacks"

_cic_sf_all = glob.glob(os.path.join(BASE_DIR, "**", "stateful_features*.csv"),
                         recursive=True)
_cic_sl_all = glob.glob(os.path.join(BASE_DIR, "**", "stateless_features*.csv"),
                         recursive=True)

cic_sf_files = [f for f in _cic_sf_all if _is_cic_attack(f)]
cic_sl_files = [f for f in _cic_sl_all if _is_cic_attack(f)]

mut_sf_files  = glob.glob(os.path.join(BASE_DIR, "**", "*bridge*stateful*.csv"),
                           recursive=True)
mut_sl_files  = glob.glob(os.path.join(BASE_DIR, "**", "*bridge*stateless*.csv"),
                           recursive=True)

# Alternative: search by Mutant Payload pattern (ARAGAT generated)
if not mut_sf_files:
    mut_sf_files = glob.glob(os.path.join(BASE_DIR, "**", "stateful_features-bridge*"),
                              recursive=True)
    mut_sl_files = glob.glob(os.path.join(BASE_DIR, "**", "stateless_features-bridge*"),
                              recursive=True)

print(f"\n📂 CIC-2021 total encontrados — stateful: {len(_cic_sf_all)}, stateless: {len(_cic_sl_all)}")
print(f"📂 CIC-2021 ATAQUE (Attacks/) — stateful: {len(cic_sf_files)}, stateless: {len(cic_sl_files)}")
print(f"📂 Mutant Payload             — stateful: {len(mut_sf_files)}, stateless: {len(mut_sl_files)}")
if len(cic_sf_files) == 0:
    raise FileNotFoundError(
        f"\n❌ No se encontraron archivos CIC en carpetas 'Attacks/'.\n"
        f"   Verifica que BASE_DIR sea correcto: {BASE_DIR}\n"
        f"   Y que la estructura sea: .../Attacks/stateful_features-*.csv"
    )

print(f"\n✅ Activation sub-model created (up to '{last_conv.name}')")


# =============================================================================
# STEP 1: FUNCTION TO COLLECT N FLOWS
# =============================================================================

def collect_attack_flows(sf_files, sl_files, model, scaler,
                         n_flows=100, is_cic=True, min_prob=0.5,
                         label_col='label'):
    """
    Collects up to n_flows attack flows with prediction >= min_prob.

    Parameters
    ----------
    sf_files   : lista de rutas stateful CSV
    sl_files   : lista de rutas stateless CSV (misma longitud y orden)
    model      : modelo CNN completo (para filtrar por prob)
    scaler     : StandardScaler ya ajustado
    n_flows    : maximum number of flows to collect
    is_cic     : True = CIC 2021 (ataques en carpeta "Attacks"),
                 False = Mutant Payload (ataques tienen label==1)
    min_prob   : minimum prediction probability threshold
    label_col  : nombre de columna de etiqueta en CSV

    Returns
    -------
    x_list  : lista de arrays (1, 23, 1)
    probs   : lista de probabilidades correspondientes
    """
    x_list = []
    probs  = []
    total_inspected = 0

    # Alternative label column names (CIC uses 'Label' with uppercase L)
    LABEL_CANDIDATES = [label_col, label_col.capitalize(), label_col.upper(),
                        'Label', 'label', 'class', 'Class', 'is_attack', 'type']

    def _find_label_col(df):
        """Retorna el nombre de la columna de etiqueta encontrada, o None."""
        for c in LABEL_CANDIDATES:
            if c in df.columns:
                return c
        return None

    # Emparejar stateful con stateless usando el nombre de archivo base
    sf_map = {os.path.basename(p).replace("stateful", "PLACEHOLDER"): p
              for p in sf_files}
    sl_map = {os.path.basename(p).replace("stateless", "PLACEHOLDER"): p
              for p in sl_files}
    paired_keys = set(sf_map.keys()) & set(sl_map.keys())

    if not paired_keys:
        # Fallback: intentar emparejar por orden de lista
        pairs = list(zip(sorted(sf_files), sorted(sl_files)))
        print(f"   ⚠️  Name-based pairing failed, using order ({len(pairs)} pairs)")
    else:
        pairs = [(sf_map[k], sl_map[k]) for k in sorted(paired_keys)]
        print(f"   ✅ {len(pairs)} pares stateful/stateless emparejados por nombre")

    # Diagnostics: show columns of first pair to detect label column name
    if pairs:
        _df0 = pd.read_csv(pairs[0][0])
        _lc  = _find_label_col(_df0) if len(pairs) > 0 else None
        _parent0 = os.path.basename(os.path.dirname(pairs[0][0]))
        print(f"   🔍 Diagnostics first CSV — columns: {list(_df0.columns[:8])}{'...' if len(_df0.columns) > 8 else ''}")
        print(f"   🔍 Columna label detectada: {_lc!r}  |  folder padre: '{_parent0}'")
        print(f"   🔍 path: ...{pairs[0][0][-70:]}")

    _first_error_shown = False   # muestra el 1er error de clean_and_prepare (global)
    _diag_done = False           # column diagnostics only once

    for sf_path, sl_path in pairs:
        if len(x_list) >= n_flows:
            break
        try:
            df_sf = pd.read_csv(sf_path)
            df_sl = pd.read_csv(sl_path)

            # Para CIC: todos los archivos ya fueron pre-filtrados a "Attacks/" en PASO 0.
            # Para Mutant: se filtra por columna de etiqueta.
            if is_cic:
                rows_to_use = range(len(df_sf))
            else:
                # Mutant Payload: filtrar por label interno (auto-detecta nombre de col)
                found_col = _find_label_col(df_sf) or _find_label_col(df_sl)
                if found_col is None:
                    rows_to_use = range(len(df_sf))   # sin label → asumir que es ataque
                else:
                    src = df_sf if found_col in df_sf.columns else df_sl
                    attack_rows = src.index[src[found_col] == 1].tolist()
                    if not attack_rows:
                        continue
                    rows_to_use = attack_rows

            # Missing column diagnostics (only for the first processed pair)
            if not _diag_done:
                _diag_done = True
                # total_queries se calcula en clean_and_prepare, no viene del CSV
                computed_cols = {'total_queries'}
                missing_sf = [c for c in FEATS_SF
                              if c not in df_sf.columns and c not in computed_cols]
                missing_sl = [c for c in FEATS_SL if c not in df_sl.columns]
                if missing_sf:
                    print(f"   ⚠️  Columnas FEATS_SF faltantes (no computadas): {missing_sf}")
                if missing_sl:
                    print(f"   ⚠️  Columnas FEATS_SL faltantes en stateless CSV: {missing_sl}")
                if not missing_sf and not missing_sl:
                    print(f"   ✅ Todas las columnas de FEATS_SF/FEATS_SL presentes (o computadas).")

            n_before = len(x_list)
            for i in rows_to_use:
                if len(x_list) >= n_flows:
                    break
                if i >= len(df_sf):
                    continue
                try:
                    x_in = clean_and_prepare(df_sl, df_sf, i, scaler)
                    total_inspected += 1
                    prob = float(model.predict(x_in, verbose=0)[0][0])
                    if prob >= min_prob:
                        x_list.append(x_in)
                        probs.append(prob)
                except Exception as _e:
                    if not _first_error_shown:
                        print(f"   ❌ Error en clean_and_prepare (fila {i}): "
                              f"{type(_e).__name__}: {_e}")
                        _first_error_shown = True
                    continue

            n_added = len(x_list) - n_before
            if n_added > 0:
                fname = os.path.basename(sf_path)
                print(f"   ├─ {fname}: +{n_added} flows "
                      f"(total: {len(x_list)}/{n_flows})")

        except Exception as e:
            print(f"   ⚠️  Error en {os.path.basename(sf_path)}: {e}")
            continue

    print(f"   └─ Inspectados: {total_inspected} flows | "
          f"Valid (prob≥{min_prob}): {len(x_list)}")

    if len(x_list) < 10:
        print(f"\n   ⚠️  WARNING: Only {len(x_list)} valid flows found.")
        print(f"   Considera bajar min_prob (actual={min_prob}) o revisar rutas.")
        if len(x_list) == 0:
            raise ValueError("❌ No valid flows found. "
                             "Verifica rutas y estructura de archivos.")

    return x_list, probs


# =============================================================================
# STEP 2: REAL AVERAGE PROFILE COMPUTATION (with error band)
# =============================================================================

def get_mean_profile_with_std(x_list, activation_model):
    """
    Computes mean activation profile over multiple flows.

    Proceso por flow:
      1. Extract activations from the first Conv1D → shape (D, F)
      2. Mean absolute activation per spatial position → shape (D,)
         (promedia sobre los F filtros)
      3. Normalize by the individual flow maximum

    Luego promedia todos los perfiles normalizados.

    Returns
    -------
    mean_p : array (D,) normalizado en [0, 1]
    std_p  : array (D,) standard deviation across flows
    n_used : int, number of processed flows
    """
    profiles = []
    for x in x_list:
        acts = activation_model.predict(x, verbose=0)[0]  # (D, F)
        p    = np.mean(np.abs(acts), axis=1)               # (D,)  mean sobre filtros
        norm = p.max() + 1e-9
        profiles.append(p / norm)

    profiles = np.array(profiles)           # (N, D)
    mean_p   = np.mean(profiles, axis=0)   # (D,)
    std_p    = np.std(profiles, axis=0)    # (D,)

    # Re-normalizar el promedio final a [0, 1]
    mean_p = mean_p / (mean_p.max() + 1e-9)

    return mean_p, std_p, len(profiles)


# =============================================================================
# FLOW COLLECTION
# =============================================================================

N_FLOWS      = 100   # flows objetivo por panel
MIN_PROB     = 0.5   # umbral para CIC-2021 (modelo detecta bien → exigir prob alta)
MIN_PROB_MUT = 0.0   # umbral para Mutant Payload (modelo FALLA → prob < 0.5 esperada;
                     # use all flows with label==1 to show real activation)

print("\n" + "=" * 60)
print("PASO 1: RECOLECTANDO FLOWS CIC-2021")
print("=" * 60)
x_cic_list, probs_cic = collect_attack_flows(
    sf_files  = cic_sf_files,
    sl_files  = cic_sl_files,
    model     = cnn_model,
    scaler    = scaler,
    n_flows   = N_FLOWS,
    is_cic    = True,
    min_prob  = MIN_PROB
)

print("\n" + "=" * 60)
print("PASO 1b: RECOLECTANDO FLOWS MUTANT PAYLOAD")
print("=" * 60)
x_mut_list, probs_mut = collect_attack_flows(
    sf_files  = mut_sf_files,
    sl_files  = mut_sl_files,
    model     = cnn_model,
    scaler    = scaler,
    n_flows   = N_FLOWS,
    is_cic    = False,
    min_prob  = MIN_PROB_MUT   # 0.0: aceptar todos los flows con label==1
)

print("\n" + "=" * 60)
print("PASO 2: CALCULANDO PERFILES PROMEDIO")
print("=" * 60)

mean_cic, std_cic, n_cic = get_mean_profile_with_std(x_cic_list, act_model)
mean_mut, std_mut, n_mut = get_mean_profile_with_std(x_mut_list, act_model)

print(f"✅ Perfil CIC-2021:     {n_cic} flows, D={len(mean_cic)} puntos espaciales")
print(f"✅ Perfil Mutant Pay.:  {n_mut} flows, D={len(mean_mut)} puntos espaciales")
print(f"   CIC   — mean prob: {np.mean(probs_cic):.3f} ± {np.std(probs_cic):.3f}")
print(f"   Mutant — mean prob: {np.mean(probs_mut):.3f} ± {np.std(probs_mut):.3f}")

# X-axis: feature index (0–22), mapped to 0.0–1.0
# With the first Conv1D and padding='same', position i maps to feature i.
# FEATS_SL = features 0–10 (11 stateless)
# FEATS_SF = features 11–22 (12 stateful)
x_axis = np.linspace(0.0, 1.0, D)

# Exact boundary between FEATS_SL and FEATS_SF on the normalized axis
N_SL = len(FEATS_SL)   # 11
N_SF = len(FEATS_SF)   # 12
N_TOTAL = N_SL + N_SF  # 23

# La metadata de CIC (bytes 0-20) afecta principalmente FEATS_SL (stateless):
# subdomain_length, entropy, FQDN_count... → primeras N_SL posiciones
# La metadata de Mutant (bytes 70-80) afecta principalmente FEATS_SF (stateful):
# distinct_ip, unique_ttl, rr_count... → last N_SF positions
sl_end   = (N_SL - 0.5) / (N_TOTAL - 1)   # frontera SL/SF ≈ 0.45
sf_start = (N_SL + 0.5) / (N_TOTAL - 1)   # inicio zona SF ≈ 0.50

# Alias for compatibility with the plotting code
metadata_frac_low  = sl_end    # shading for stateless region (panel a)
metadata_frac_high = sf_start  # shading for stateful region  (panel b)

print(f"\n📏 Mapping features → eje X normalizado:")
print(f"   FEATS_SL (0–{N_SL-1}): x ∈ [0.00, {sl_end:.2f}]  — stateless (DNS name patterns)")
print(f"   FEATS_SF ({N_SL}–{N_TOTAL-1}): x ∈ [{sf_start:.2f}, 1.00]  — stateful (flow statistics)")


# =============================================================================
# PASO 3: FIGURA MEJORADA
# =============================================================================

print("\n" + "=" * 60)
print("PASO 3: GENERANDO FIGURA")
print("=" * 60)

plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8.5), sharex=True)
fig.subplots_adjust(top=0.93, bottom=0.12, hspace=0.42, left=0.10, right=0.96)

DARK_BLUE  = '#2c3e50'
ORANGE     = '#e67e22'
RED_META   = '#c0392b'

# ─────────────────────────────────────────────────────────
# Panel (a): CIC-2021
# ─────────────────────────────────────────────────────────
ax1.plot(x_axis, mean_cic,
         color=DARK_BLUE, linewidth=2.5, marker='o', markersize=6,
         label=f'Mean profile (N={n_cic} flows)', zorder=3)
ax1.fill_between(x_axis,
                 mean_cic - std_cic, mean_cic + std_cic,
                 alpha=0.20, color=DARK_BLUE, label='±1 SD', zorder=2)
ax1.axvspan(0, metadata_frac_low,
            alpha=0.12, color=RED_META, zorder=1)
ax1.axvspan(sf_start, 1.0,
            alpha=0.10, color=ORANGE, zorder=1)

# Peak annotation — the actual peak lies in the stateful region (right)
peak_idx_cic = int(np.argmax(mean_cic))
peak_x_cic   = x_axis[peak_idx_cic]
ann_x_cic    = max(peak_x_cic - 0.42, 0.10)   # texto a la izquierda del pico
ax1.annotate(
    'Strong activation in\nstateful features\n(flow statistics)',
    xy=(peak_x_cic, mean_cic[peak_idx_cic]),
    xytext=(ann_x_cic, 0.75),
    fontsize=9, ha='center', va='top',
    color=DARK_BLUE, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=DARK_BLUE, lw=1.5,
                    connectionstyle='arc3,rad=-0.2'),
    bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
              edgecolor=DARK_BLUE, alpha=0.90, lw=0.8)
)

ax1.set_title('(a) Standard Attack (CIC-2021): Metadata at byte offset 0–20',
              fontweight='bold', fontsize=11, pad=8)
# SL / SF boundary line
ax1.axvline(x=sl_end, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax1.set_ylabel('Normalized Activation Magnitude', fontsize=10)
ax1.set_ylim(-0.05, 1.30)
ax1.legend(loc='upper left', fontsize=9, framealpha=0.85)
ax1.tick_params(labelsize=9)

# ─────────────────────────────────────────────────────────
# Panel (b): Mutant Payload
# ─────────────────────────────────────────────────────────
ax2.plot(x_axis, mean_mut,
         color=ORANGE, linewidth=2.5, marker='s', markersize=6,
         label=f'Mean profile (N={n_mut} flows)', zorder=3)
ax2.fill_between(x_axis,
                 mean_mut - std_mut, mean_mut + std_mut,
                 alpha=0.20, color=ORANGE, label='±1 SD', zorder=2)
ax2.axvspan(0, sl_end,
            alpha=0.10, color=RED_META, zorder=1)
ax2.axvspan(metadata_frac_high, 1.0,
            alpha=0.12, color=ORANGE, zorder=1)

# Peak annotation — text positioned to avoid legend overlap (upper right)
peak_idx_mut = int(np.argmax(mean_mut))
peak_x_mut   = x_axis[peak_idx_mut]
# If peak is in the right half, place text on the left and vice versa
if peak_x_mut >= 0.5:
    ann_x_mut = peak_x_mut - 0.38
    rad = 0.20
else:
    ann_x_mut = peak_x_mut + 0.38
    rad = -0.20
ax2.annotate(
    'Peak in stateful features\n(flow statistics)',
    xy=(peak_x_mut, mean_mut[peak_idx_mut]),
    xytext=(ann_x_mut, 0.68),
    fontsize=9, ha='center', va='top',
    color=ORANGE, fontweight='bold',
    arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.5,
                    connectionstyle=f'arc3,rad={rad}'),
    bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
              edgecolor=ORANGE, alpha=0.90, lw=0.8)
)

ax2.set_title('(b) Evasive Attack (Mutant Payload): Metadata shifted to byte offset 70–80',
              fontweight='bold', fontsize=11, pad=8)
# SL / SF boundary line
ax2.axvline(x=sf_start, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax2.set_ylabel('Normalized Activation Magnitude', fontsize=10)
ax2.set_xlabel('Feature Sequence Position (normalized)', fontsize=10)
ax2.set_ylim(-0.05, 1.30)
ax2.set_xlim(-0.02, 1.02)
ax2.legend(loc='upper right', fontsize=9, framealpha=0.85)
ax2.tick_params(labelsize=9)

# ─────────────────────────────────────────────────────────
# Leyenda global compartida — debajo del panel (b)
# ─────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(facecolor=RED_META, alpha=0.35,
                   label='Stateless features — DNS name patterns (FEATS_SL, pos. 0–10)'),
    mpatches.Patch(facecolor=ORANGE,   alpha=0.35,
                   label='Stateful features — flow statistics (FEATS_SF, pos. 11–22)'),
]
fig.legend(handles=legend_patches,
           loc='lower center', ncol=2,
           fontsize=8.5, framealpha=0.85,
           bbox_to_anchor=(0.5, 0.00))

fig.suptitle(
    'CNN Feature Group Activation: Stateless vs. Stateful Response',
    fontsize=12, fontweight='bold'
)


# =============================================================================
# PASO 4: MANEJO DE CASOS EDGE — reporte de advertencias en el plot
# =============================================================================

warnings = []
if D == 2:
    warnings.append(f"⚠ Solo D={D} puntos espaciales (arquitectura con MaxPooling agresivo)")
if n_cic < 10:
    warnings.append(f"⚠ Pocos flows CIC-2021: N={n_cic} (recomendado ≥10)")
if n_mut < 10:
    warnings.append(f"⚠ Pocos flows Mutant Payload: N={n_mut} (recomendado ≥10)")

if warnings:
    warning_text = "\n".join(warnings)
    fig.text(0.01, 0.01, warning_text, fontsize=7, color='#cc0000',
             va='bottom', ha='left',
             bbox=dict(boxstyle='round', facecolor='#fff3f3',
                       edgecolor='#cc0000', alpha=0.8))


# =============================================================================
# PASO 5: GUARDAR
# =============================================================================

plt.tight_layout(rect=[0, 0.06, 1, 0.97])

out_png = "figure2_cnn_activation_FINAL.png"
out_pdf = "figure2_cnn_activation_FINAL.pdf"

plt.savefig(out_png, dpi=300, bbox_inches='tight')
plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "=" * 60)
print("FIGURE 2 GENERATION REPORT")
print("=" * 60)
print(f"CNN last Conv1D output shape : (None, {D}, {F})")
print(f"Spatial dimensions (D)       : {D}")
print(f"Filters (F)                  : {F}")
print(f"CIC-2021 flows used          : {n_cic}  (mean prob={np.mean(probs_cic):.3f})")
print(f"Mutant Payload flows used    : {n_mut}  (mean prob={np.mean(probs_mut):.3f})")
print(f"Figura PNG guardada          : {out_png}")
print(f"Figura PDF guardada          : {out_pdf}")
if warnings:
    print("\nADVERTENCIAS:")
    for w in warnings:
        print(f"  {w}")
print("=" * 60)


# =============================================================================
# PASO 6: CAPTION LaTeX LISTO PARA COPIAR
# (reemplazar N_CIC y N_MUT con los valores del reporte)
# =============================================================================
caption_latex = r"""
% ============================================================
% CAPTION PARA LATEX — copiar al paper
% ============================================================
\begin{figure}[t]
  \centering
  \includegraphics[width=\columnwidth]{figure2_cnn_activation_FINAL.pdf}
  \caption{Normalized mean activation profiles of the \emph{first}
  convolutional layer (64 filters, kernel size~3) averaged over
  100~CIC-2021 attack flows \textbf{(a)} and 100~Mutant~Payload
  evasive attack flows \textbf{(b)} ($\pm$1\,SD shaded). Features
  are grouped as stateless (DNS name patterns, positions~0--10,
  pink region) and stateful (flow statistics, positions~11--22,
  orange region). In \textbf{(a)}, standard attacks produce
  moderate activation across stateless features and strong
  activation in stateful features. In \textbf{(b)}, stateless
  activation collapses nearly to zero when metadata is shifted
  to byte offset~70--80, while stateful features maintain partial
  activation---explaining the CNN's partial robustness (29.85\%
  recall): stateful flow statistics preserve discriminative power
  even when DNS name patterns are disrupted by positional mutation.}
  \label{fig:cnn_activation}
\end{figure}
% ============================================================
"""
print(caption_latex)


## Experiment 3: Ablation Study — Benign Shift vs Attack Mutation

Disentangle two sources of recall degradation:
- **Scenario A**: CIC-2021 baseline (no distribution shift)
- **Scenario B**: Benign traffic shift (same attack, shifted benign)
- **Scenario C**: Attack mutation (same benign, mutated attack payload)

In [ ]:
# =============================================================================
# EXPERIMENT 3: ABLATION STUDY - BENIGN SHIFT VS ATTACK MUTATION
# =============================================================================
# CORRECTED VERSION - Consistency with Experiment 2
# - Seeds unificados para benign y attack
# - Test sets guardados para reproducibilidad
# - Mismo test set compartido entre Exp 2 y Exp 3 Scenario C
# =============================================================================

import os
import glob
import pandas as pd
import numpy as np
import joblib
from tensorflow import keras
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score
from sklearn.model_selection import train_test_split

# =============================================================================
# CONFIGURATION
# =============================================================================

ARAGAT_DIR = "/content/drive/MyDrive/Tunnel/CSV_CIC21/CSV_Generated"
MODEL_DIR_SOTA = "/content/drive/MyDrive/Tunnel/CSV_CIC21/Models_SOTA_Hybrid"
CIC_BASE = "/content/drive/MyDrive/Tunnel/CSV_CIC21/CSV_CIC21"

# Directory to save test sets (for reuse in Experiment 2)
TESTSET_DIR = "/content/drive/MyDrive/Tunnel/CSV_CIC21/Test_Sets"
os.makedirs(TESTSET_DIR, exist_ok=True)

FEATS_SL = [
    "FQDN_count", "subdomain_length", "upper", "lower", "numeric",
    "entropy", "special", "labels", "labels_max", "labels_average", "len"
]

FEATS_SF = [
    "rr", "A_frequency", "AAAA_frequency", "CNAME_frequency", "TXT_frequency",
    "MX_frequency", "NS_frequency", "NULL_frequency",
    "rr_count", "distinct_ip", "unique_ttl", "total_queries"
]

ALL_FEATS = FEATS_SL + FEATS_SF

# SEEDS UNIFICADOS - Mismos para benign y attack
SHARED_SEEDS = [10, 21, 35, 42, 55, 60, 75, 82, 99]

# Test set size (balanced)
TEST_SIZE_PER_CLASS = 2500  # 2500 benign + 2500 malicious = 5000 total

# =============================================================================
# FUNCIONES DE CARGA
# =============================================================================

def load_cic_attacks(verbose=True):
    """
    Load attack flows from CIC-2021 (fixed position 0-20%)
    Returns: X_attacks, y_attacks
    """
    CIC_PATHS = {
        'Light_Attack': os.path.join(CIC_BASE, "Attack_Light_Benign/Attacks"),
        'Heavy_Attack': os.path.join(CIC_BASE, "Attack_heavy_Benign/Attacks")
    }

    combined_frames = []

    for label_type, folder_path in CIC_PATHS.items():
        if not os.path.exists(folder_path):
            continue

        sl_files = glob.glob(os.path.join(folder_path, "stateless_*.csv"))

        for sl_path in sl_files:
            try:
                df_sl = pd.read_csv(sl_path)
                if df_sl.empty:
                    continue

                filename = os.path.basename(sl_path)

                if "light_benign" in filename:
                    sf_filename = filename.replace(
                        "stateless_features-light_benign",
                        "stateful_features-_light_benign"
                    )
                else:
                    sf_filename = filename.replace("stateless_", "stateful_")

                sf_path = os.path.join(folder_path, sf_filename)

                if not os.path.exists(sf_path):
                    candidates = glob.glob(os.path.join(folder_path, "stateful_*.csv"))
                    core_name = filename.replace("stateless_features-", "").replace(".pcap.csv", "")
                    match = next((c for c in candidates if core_name in c), None)
                    if match:
                        sf_path = match
                    else:
                        continue

                df_sf = pd.read_csv(sf_path)

                min_len = min(len(df_sl), len(df_sf))
                if min_len == 0:
                    continue

                df_sl = df_sl.iloc[:min_len].reset_index(drop=True)
                df_sf = df_sf.iloc[:min_len].reset_index(drop=True)

                df_sl = df_sl.reindex(columns=FEATS_SL, fill_value=0)
                for c in FEATS_SL:
                    df_sl[c] = pd.to_numeric(df_sl[c], errors='coerce').fillna(0)

                df_sf = df_sf.reindex(columns=FEATS_SF, fill_value=0)
                for c in FEATS_SF:
                    df_sf[c] = pd.to_numeric(df_sf[c], errors='coerce').fillna(0)

                df_hybrid = pd.concat([df_sl[FEATS_SL], df_sf[FEATS_SF]], axis=1)
                combined_frames.append(df_hybrid)

            except Exception as e:
                if verbose:
                    print(f"   ⚠️  Error loading {sl_path}: {e}")
                pass

    if not combined_frames:
        if verbose:
            print("   ⚠️  CIC-2021 attacks could not be loaded")
        return np.array([]), np.array([])

    df_all = pd.concat(combined_frames, ignore_index=True)
    X_all = df_all[ALL_FEATS].values
    y_all = np.ones(len(X_all), dtype=int)

    if verbose:
        print(f"   ✅ CIC-2021 attacks loaded: {len(X_all):,} flows")

    return X_all, y_all

def load_cic_benign(verbose=True):
    """
    Carga flows benignos de CIC-2021
    Returns: X_benign, y_benign
    """
    CIC_BENIGN_PATHS = {
        'Light_Benign': os.path.join(CIC_BASE, "Attack_Light_Benign/Benign"),
        'Heavy_Benign': os.path.join(CIC_BASE, "Attack_heavy_Benign/Benign")
    }

    combined_frames = []

    for label_type, folder_path in CIC_BENIGN_PATHS.items():
        if not os.path.exists(folder_path):
            continue

        sl_files = glob.glob(os.path.join(folder_path, "stateless_*.csv"))

        for sl_path in sl_files:
            try:
                df_sl = pd.read_csv(sl_path)
                if df_sl.empty:
                    continue

                filename = os.path.basename(sl_path)
                sf_filename = filename.replace("stateless_", "stateful_")
                sf_path = os.path.join(folder_path, sf_filename)

                if not os.path.exists(sf_path):
                    candidates = glob.glob(os.path.join(folder_path, "stateful_*.csv"))
                    core_name = filename.replace("stateless_features-", "").replace(".pcap.csv", "")
                    match = next((c for c in candidates if core_name in c), None)
                    if match:
                        sf_path = match
                    else:
                        continue

                df_sf = pd.read_csv(sf_path)

                min_len = min(len(df_sl), len(df_sf))
                if min_len == 0:
                    continue

                df_sl = df_sl.iloc[:min_len].reset_index(drop=True)
                df_sf = df_sf.iloc[:min_len].reset_index(drop=True)

                df_sl = df_sl.reindex(columns=FEATS_SL, fill_value=0)
                for c in FEATS_SL:
                    df_sl[c] = pd.to_numeric(df_sl[c], errors='coerce').fillna(0)

                df_sf = df_sf.reindex(columns=FEATS_SF, fill_value=0)
                for c in FEATS_SF:
                    df_sf[c] = pd.to_numeric(df_sf[c], errors='coerce').fillna(0)

                df_hybrid = pd.concat([df_sl[FEATS_SL], df_sf[FEATS_SF]], axis=1)
                combined_frames.append(df_hybrid)

            except Exception as e:
                if verbose:
                    print(f"   ⚠️  Error loading {sl_path}: {e}")
                pass

    if not combined_frames:
        if verbose:
            print("   ⚠️  CIC-2021 benign could not be loaded")
        return np.array([]), np.array([])

    df_all = pd.concat(combined_frames, ignore_index=True)
    X_all = df_all[ALL_FEATS].values
    y_all = np.zeros(len(X_all), dtype=int)

    if verbose:
        print(f"   ✅ CIC-2021 benign loaded: {len(X_all):,} flows")

    return X_all, y_all

def load_mutant_attacks(seeds=SHARED_SEEDS, verbose=True):
    """
    Load attack flows from Mutant Payload (mutated position 70-80%)
    Args:
        seeds: Lista de seeds a cargar
    Returns: X_attacks, y_attacks
    """
    all_flows = []

    for seed in seeds:
        # Cargar metadata
        meta_path = os.path.join(ARAGAT_DIR, f"dns_tunnels_metadata_{seed}.csv")
        if not os.path.exists(meta_path):
            if verbose:
                print(f"   ⚠️  Metadata not found for seed {seed}")
            continue

        df_meta = pd.read_csv(meta_path)

        # Filtrar solo ataques (label=1)
        df_attacks = df_meta[df_meta['label'] == 1]

        if len(df_attacks) == 0:
            if verbose:
                print(f"   ⚠️  No attacks found in seed {seed}")
            continue

        # Cargar stateless ENRICHED
        sl_path = os.path.join(ARAGAT_DIR, f"stateless_features_enriched_{seed}.csv")
        if not os.path.exists(sl_path):
            if verbose:
                print(f"   ⚠️  Stateless features not found for seed {seed}")
            continue

        df_sl = pd.read_csv(sl_path)

        # Agregar a nivel de flow
        aggregation = {feat: 'mean' for feat in FEATS_SL if feat in df_sl.columns}
        df_sl_agg = df_sl.groupby('tunnel_id').agg(aggregation).reset_index()

        # Cargar stateful ENRICHED
        sf_path = os.path.join(ARAGAT_DIR, f"stateful_features_enriched_{seed}.csv")
        if not os.path.exists(sf_path):
            if verbose:
                print(f"   ⚠️  Stateful features not found for seed {seed}")
            continue

        df_sf = pd.read_csv(sf_path)

        # Merge
        df_merged = pd.merge(df_sl_agg, df_sf, on='tunnel_id', how='inner', suffixes=('', '_sf'))

        # Filtrar solo tunnel_ids de ataques
        attack_ids = df_attacks['tunnel_id'].values
        df_filtered = df_merged[df_merged['tunnel_id'].isin(attack_ids)].copy()

        if len(df_filtered) == 0:
            if verbose:
                print(f"   ⚠️  No attack flows after filtering for seed {seed}")
            continue

        # Alinear features
        df_filtered = df_filtered.reindex(columns=ALL_FEATS, fill_value=0)
        df_filtered[ALL_FEATS] = df_filtered[ALL_FEATS].replace([np.inf, -np.inf], np.nan).fillna(0)

        all_flows.append(df_filtered)

        if verbose:
            print(f"   📌 Seed {seed}: {len(df_filtered)} attack flows")

    if not all_flows:
        if verbose:
            print("   ❌ Mutant Payload attacks could not be loaded from any seed")
        return np.array([]), np.array([])

    df_combined = pd.concat(all_flows, ignore_index=True)
    X = df_combined[ALL_FEATS].values
    y = np.ones(len(X), dtype=int)

    if verbose:
        print(f"   ✅ Mutant Payload attacks loaded: {len(X):,} flows (from {len(seeds)} seeds)")

    return X, y

def load_mutant_benign(seeds=SHARED_SEEDS, verbose=True):
    """
    Carga flows benignos de Mutant Payload
    Args:
        seeds: Lista de seeds a cargar
    Returns: X_benign, y_benign
    """
    all_flows = []

    for seed in seeds:
        # Cargar metadata
        meta_path = os.path.join(ARAGAT_DIR, f"dns_tunnels_metadata_{seed}.csv")
        if not os.path.exists(meta_path):
            if verbose:
                print(f"   ⚠️  Metadata not found for seed {seed}")
            continue

        df_meta = pd.read_csv(meta_path)

        # Filtrar solo benignos (label=0)
        df_benign = df_meta[df_meta['label'] == 0]

        if len(df_benign) == 0:
            if verbose:
                print(f"   ⚠️  No benign flows found in seed {seed}")
            continue

        # Cargar stateless ENRICHED
        sl_path = os.path.join(ARAGAT_DIR, f"stateless_features_enriched_{seed}.csv")
        if not os.path.exists(sl_path):
            if verbose:
                print(f"   ⚠️  Stateless features not found for seed {seed}")
            continue

        df_sl = pd.read_csv(sl_path)

        # Agregar a nivel de flow
        aggregation = {feat: 'mean' for feat in FEATS_SL if feat in df_sl.columns}
        df_sl_agg = df_sl.groupby('tunnel_id').agg(aggregation).reset_index()

        # Cargar stateful ENRICHED
        sf_path = os.path.join(ARAGAT_DIR, f"stateful_features_enriched_{seed}.csv")
        if not os.path.exists(sf_path):
            if verbose:
                print(f"   ⚠️  Stateful features not found for seed {seed}")
            continue

        df_sf = pd.read_csv(sf_path)

        # Merge
        df_merged = pd.merge(df_sl_agg, df_sf, on='tunnel_id', how='inner', suffixes=('', '_sf'))

        # Filtrar solo tunnel_ids benignos
        benign_ids = df_benign['tunnel_id'].values
        df_filtered = df_merged[df_merged['tunnel_id'].isin(benign_ids)].copy()

        if len(df_filtered) == 0:
            if verbose:
                print(f"   ⚠️  No benign flows after filtering for seed {seed}")
            continue

        # Alinear features
        df_filtered = df_filtered.reindex(columns=ALL_FEATS, fill_value=0)
        df_filtered[ALL_FEATS] = df_filtered[ALL_FEATS].replace([np.inf, -np.inf], np.nan).fillna(0)

        all_flows.append(df_filtered)

        if verbose:
            print(f"   📌 Seed {seed}: {len(df_filtered)} benign flows")

    if not all_flows:
        if verbose:
            print("   ❌ Mutant Payload benign could not be loaded from any seed")
        return np.array([]), np.array([])

    df_combined = pd.concat(all_flows, ignore_index=True)
    X = df_combined[ALL_FEATS].values
    y = np.zeros(len(X), dtype=int)

    if verbose:
        print(f"   ✅ Mutant Payload benign loaded: {len(X):,} flows (from {len(seeds)} seeds)")

    return X, y

def create_balanced_test_set(X_benign, y_benign, X_malicious, y_malicious,
                             n_per_class=2500, random_state=42):
    """
    Crea test set balanceado con n_per_class samples de cada clase
    Args:
        X_benign, y_benign: Arrays de flows benignos
        X_malicious, y_malicious: Arrays de flows maliciosos
        n_per_class: Number of samples per class
        random_state: Seed para reproducibilidad
    Returns:
        X_test, y_test: Test set balanceado y shuffled
    """
    rng = np.random.RandomState(random_state)

    # Validar que hay suficientes samples
    if len(X_benign) < n_per_class:
        print(f"   ⚠️  WARNING: Only {len(X_benign)} benign flows available (need {n_per_class})")
        n_benign = len(X_benign)
    else:
        n_benign = n_per_class

    if len(X_malicious) < n_per_class:
        print(f"   ⚠️  WARNING: Only {len(X_malicious)} malicious flows available (need {n_per_class})")
        n_malicious = len(X_malicious)
    else:
        n_malicious = n_per_class

    # Sample benign
    if len(X_benign) >= n_benign:
        indices_benign = rng.choice(len(X_benign), size=n_benign, replace=False)
        X_ben_sample = X_benign[indices_benign]
        y_ben_sample = y_benign[indices_benign]
    else:
        X_ben_sample = X_benign
        y_ben_sample = y_benign

    # Sample malicious
    if len(X_malicious) >= n_malicious:
        indices_mal = rng.choice(len(X_malicious), size=n_malicious, replace=False)
        X_mal_sample = X_malicious[indices_mal]
        y_mal_sample = y_malicious[indices_mal]
    else:
        X_mal_sample = X_malicious
        y_mal_sample = y_malicious

    # Combinar
    X_test = np.vstack([X_ben_sample, X_mal_sample])
    y_test = np.hstack([y_ben_sample, y_mal_sample])

    # Shuffle
    indices = rng.permutation(len(X_test))
    X_test = X_test[indices]
    y_test = y_test[indices]

    return X_test, y_test

def evaluate_model(model, X, y, model_name, is_dl=False, scaler=None):
    """
    Evaluates model and returns metrics
    Args:
        model: Modelo entrenado
        X, y: Datos de test
        model_name: Nombre del modelo
        is_dl: Si es modelo de deep learning
        scaler: Scaler for normalization
    Returns:
        Dict with metrics
    """
    if len(X) == 0:
        return {
            'accuracy': np.nan,
            'precision': np.nan,
            'recall': np.nan,
            'f1': np.nan
        }

    # Preparar datos
    if model_name == 'LogisticRegression' or is_dl:
        X_eval = scaler.transform(X)
        if is_dl:
            X_eval = X_eval.reshape((X_eval.shape[0], X_eval.shape[1], 1))
            y_pred_proba = model.predict(X_eval, verbose=0).flatten()
            y_pred = (y_pred_proba > 0.5).astype(int)
        else:
            y_pred = model.predict(X_eval)
    else:
        # Tree-based models: sin scaler
        y_pred = model.predict(X)

    return {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall': recall_score(y, y_pred, zero_division=0),
        'f1': f1_score(y, y_pred, zero_division=0)
    }

def save_test_set(X, y, scenario_name):
    """Saves test set for reuse"""
    filepath = os.path.join(TESTSET_DIR, f"testset_{scenario_name}.npz")
    np.savez_compressed(
        filepath,
        X=X,
        y=y,
        scenario=scenario_name,
        n_samples=len(X),
        n_benign=np.sum(y == 0),
        n_malicious=np.sum(y == 1)
    )
    print(f"   💾 Test set saved: {filepath}")
    return filepath

# =============================================================================
# MAIN EXPERIMENT
# =============================================================================

if __name__ == "__main__":
    print("\n" + "="*70)
    print("🧬 EXPERIMENT 3: ABLATION STUDY")
    print("="*70)
    print(f"\n📌 Configuration:")
    print(f"   - Shared seeds: {SHARED_SEEDS}")
    print(f"   - Test size per class: {TEST_SIZE_PER_CLASS}")
    print(f"   - Random state: 42 (fixed for reproducibility)")

    # =========================================================================
    # 1. CARGAR TODOS LOS DATASETS
    # =========================================================================

    print("\n📂 Loading datasets...")

    print("\n[1/4] Loading CIC-2021 attacks...")
    X_cic_attacks, y_cic_attacks = load_cic_attacks(verbose=True)

    print("\n[2/4] Loading CIC-2021 benign...")
    X_cic_benign, y_cic_benign = load_cic_benign(verbose=True)

    print("\n[3/4] Loading Mutant Payload attacks...")
    X_mut_attacks, y_mut_attacks = load_mutant_attacks(seeds=SHARED_SEEDS, verbose=True)

    print("\n[4/4] Loading Mutant Payload benign...")
    X_mut_benign, y_mut_benign = load_mutant_benign(seeds=SHARED_SEEDS, verbose=True)

    # Load validation
    print("\n" + "="*70)
    print("📊 Dataset Summary:")
    print("="*70)
    print(f"CIC-2021:")
    print(f"   - Attacks: {len(X_cic_attacks):,} flows")
    print(f"   - Benign:  {len(X_cic_benign):,} flows")
    print(f"\nMutant Payload:")
    print(f"   - Attacks: {len(X_mut_attacks):,} flows")
    print(f"   - Benign:  {len(X_mut_benign):,} flows")

    # =========================================================================
    # 2. CREAR 3 TEST SETS BALANCEADOS
    # =========================================================================

    print("\n" + "="*70)
    print("📊 Creating balanced test sets...")
    print("="*70)

    # Scenario A: CIC benign + CIC malicious (Baseline)
    print("\n[Scenario A] Baseline: CIC benign + CIC attacks")
    X_test_A, y_test_A = create_balanced_test_set(
        X_cic_benign, y_cic_benign,
        X_cic_attacks, y_cic_attacks,
        n_per_class=TEST_SIZE_PER_CLASS,
        random_state=42
    )
    print(f"   ✅ Created: {len(X_test_A):,} samples " +
          f"({np.sum(y_test_A == 0)} benign, {np.sum(y_test_A == 1)} malicious)")
    save_test_set(X_test_A, y_test_A, "scenario_A_baseline")

    # Scenario B: Mutant benign + CIC malicious (Benign Shift Only)
    print("\n[Scenario B] Benign Shift Only: Mutant benign + CIC attacks")
    X_test_B, y_test_B = create_balanced_test_set(
        X_mut_benign, y_mut_benign,
        X_cic_attacks, y_cic_attacks,
        n_per_class=TEST_SIZE_PER_CLASS,
        random_state=42
    )
    print(f"   ✅ Created: {len(X_test_B):,} samples " +
          f"({np.sum(y_test_B == 0)} benign, {np.sum(y_test_B == 1)} malicious)")
    save_test_set(X_test_B, y_test_B, "scenario_B_benign_shift")

    # Scenario C: Mutant benign + Mutant malicious (Full Mutation)
    # ⚠️ ESTE ES EL TEST SET QUE DEBE USARSE EN EXPERIMENTO 2
    print("\n[Scenario C] Full Mutation: Mutant benign + Mutant attacks")
    X_test_C, y_test_C = create_balanced_test_set(
        X_mut_benign, y_mut_benign,
        X_mut_attacks, y_mut_attacks,
        n_per_class=TEST_SIZE_PER_CLASS,
        random_state=42
    )
    print(f"   ✅ Created: {len(X_test_C):,} samples " +
          f"({np.sum(y_test_C == 0)} benign, {np.sum(y_test_C == 1)} malicious)")
    save_test_set(X_test_C, y_test_C, "scenario_C_full_mutation")

    print("\n" + "="*70)
    print("⚠️  IMPORTANT: Scenario C test set saved for Experiment 2 reuse")
    print(f"   Location: {TESTSET_DIR}/testset_scenario_C_full_mutation.npz")
    print("   This ensures consistency between Exp 2 and Exp 3 Scenario C")
    print("="*70)

    # =========================================================================
    # 3. CARGAR SCALER Y MODELOS
    # =========================================================================

    print("\n📦 Loading scaler and models...")
    scaler = joblib.load(os.path.join(MODEL_DIR_SOTA, "scaler_sota.joblib"))
    print(f"   ✅ Scaler loaded")

    MODEL_NAMES = ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CNN', 'LSTM']

    # =========================================================================
    # 4. EVALUAR CADA MODELO EN LOS 3 ESCENARIOS
    # =========================================================================

    results = []

    for model_name in MODEL_NAMES:
        print(f"\n{'='*70}")
        print(f"⚙️  Evaluating {model_name}")
        print(f"{'='*70}")

        # Cargar modelo
        if model_name in ['CNN', 'LSTM']:
            model_path = os.path.join(MODEL_DIR_SOTA, f"{model_name}_sota.keras")
            model = keras.models.load_model(model_path)
            is_dl = True
            print(f"   📂 Loaded: {model_name}_sota.keras")
        else:
            model_path = os.path.join(MODEL_DIR_SOTA, f"{model_name}_sota.joblib")
            model = joblib.load(model_path)
            is_dl = False
            print(f"   📂 Loaded: {model_name}_sota.joblib")

        # Evaluar en Scenario A
        print("\n   [1/3] Evaluating Scenario A (Baseline)...")
        metrics_A = evaluate_model(model, X_test_A, y_test_A, model_name, is_dl=is_dl, scaler=scaler)

        # Evaluar en Scenario B
        print("   [2/3] Evaluating Scenario B (Benign Shift Only)...")
        metrics_B = evaluate_model(model, X_test_B, y_test_B, model_name, is_dl=is_dl, scaler=scaler)

        # Evaluar en Scenario C
        print("   [3/3] Evaluating Scenario C (Full Mutation)...")
        metrics_C = evaluate_model(model, X_test_C, y_test_C, model_name, is_dl=is_dl, scaler=scaler)

        # Calcular deltas (en puntos porcentuales)
        benign_shift = (metrics_B['recall'] - metrics_A['recall']) * 100
        attack_mutation = (metrics_C['recall'] - metrics_B['recall']) * 100
        total_drop = (metrics_C['recall'] - metrics_A['recall']) * 100

        # Guardar resultados
        results.append({
            'Model': model_name,
            'Scenario_A_Recall': metrics_A['recall'],
            'Scenario_B_Recall': metrics_B['recall'],
            'Scenario_C_Recall': metrics_C['recall'],
            'Benign_Shift_pp': benign_shift,
            'Attack_Mutation_pp': attack_mutation,
            'Total_Drop_pp': total_drop
        })

        print(f"\n   📊 {model_name} Summary:")
        print(f"      Scenario A (Baseline):      {metrics_A['recall']:.2%}")
        print(f"      Scenario B (Benign Shift):  {metrics_B['recall']:.2%}")
        print(f"      Scenario C (Full Mutation): {metrics_C['recall']:.2%}")
        print(f"      Benign Shift Effect:        {benign_shift:+.2f} pp")
        print(f"      Attack Mutation Effect:     {attack_mutation:+.2f} pp")
        print(f"      Total Degradation:          {total_drop:+.2f} pp")

    # =========================================================================
    # 5. GUARDAR RESULTADOS Y GENERAR TABLA
    # =========================================================================

    df_results = pd.DataFrame(results)
    output_path = os.path.join(MODEL_DIR_SOTA, "experiment3_ablation_results.csv")
    df_results.to_csv(output_path, index=False)

    print(f"\n✅ Results saved to: {output_path}")

    # =========================================================================
    # 6. TABLA FORMATEADA PARA EL PAPER
    # =========================================================================

    print("\n" + "="*70)
    print("📊 TABLE IV - ABLATION STUDY RESULTS")
    print("="*70)

    print(f"\n{'Model':<20} {'Scenario A':>12} {'Scenario B':>12} {'Scenario C':>12} " +
          f"{'Benign':>10} {'Attack':>10} {'Total':>10}")
    print(f"{'':20} {'Baseline':>12} {'Benign Shift':>12} {'Full Mutation':>12} " +
          f"{'Shift':>10} {'Mutation':>10} {'Drop':>10}")
    print("-" * 90)

    for _, row in df_results.iterrows():
        print(f"{row['Model']:<20} {row['Scenario_A_Recall']:>11.2%} {row['Scenario_B_Recall']:>11.2%} " +
              f"{row['Scenario_C_Recall']:>11.2%} {row['Benign_Shift_pp']:>9.2f}pp {row['Attack_Mutation_pp']:>9.2f}pp " +
              f"{row['Total_Drop_pp']:>9.2f}pp")

    print("\n" + "="*70)
    print("✅ EXPERIMENT 3 COMPLETE")
    print("="*70)

    # =========================================================================
    # 7. CONSISTENCY CHECK
    # =========================================================================

    print("\n" + "="*70)
    print("🔍 CONSISTENCY VERIFICATION")
    print("="*70)

    print("\n✅ Test sets created with fixed random_state=42")
    print("✅ Seeds unified between benign and attack loads")
    print("✅ Scenario C test set saved for Experiment 2 reuse")
    print(f"\n📁 Test sets location: {TESTSET_DIR}")
    print("\n⚠️  To ensure consistency with Experiment 2:")
    print("   1. Use testset_scenario_C_full_mutation.npz in Experiment 2")
    print("   2. This eliminates sampling variance")
    print("   3. Results should now be identical between experiments")

## Experiment 4: Defense Hardening

Retrain models with augmented data (CIC-2021 + 70% ARAGAT mutants).
Evaluate on held-out 30% ARAGAT mutants (never seen during training).

In [ ]:
# =============================================================================
# EXPERIMENT 4: DEFENSE HARDENING
# =============================================================================

import os
import glob
import joblib
import pandas as pd
import numpy as np
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression

# =============================================================================
# CONFIGURATION  (same paths as Experiment 1)
# =============================================================================

BASE_DIR        = "/content/drive/MyDrive/Tunnel/CSV_CIC21"
MODEL_DIR_SOTA  = os.path.join(BASE_DIR, "Models_SOTA_Hybrid")
MODEL_DIR_HARDENED = os.path.join(BASE_DIR, "Models_Hardened")
ARAGAT_DIR      = os.path.join(BASE_DIR, "CSV_Generated")

# CIC paths — identical to Experiment 1
# (get_correct_path from Exp 1 resolves CSV_CIC21/CSV_CIC21 → CSV_CIC21)
_CIC_BASE = os.path.join(BASE_DIR, "CSV_CIC21")  # /…/CSV_CIC21/CSV_CIC21
CIC_PATHS_EXP4 = {
    'Light_Attack': os.path.join(_CIC_BASE, 'Attack_Light_Benign', 'Attacks'),
    'Light_Benign': os.path.join(_CIC_BASE, 'Attack_Light_Benign', 'Benign'),
    'Heavy_Attack': os.path.join(_CIC_BASE, 'Attack_heavy_Benign', 'Attacks'),
    'Heavy_Benign': os.path.join(_CIC_BASE, 'Attack_heavy_Benign', 'Benign'),
}

os.makedirs(MODEL_DIR_HARDENED, exist_ok=True)

# =============================================================================
# CIC LOADING — logic IDENTICAL to Experiment 1 (load_cic_hybrid_data)
# Guarantees the same distribution, the same scaler and the same baseline results
# =============================================================================

def load_cic_exp1_style(verbose=True):
    """
    Replica exacta de load_cic_hybrid_data de Experimento 1.
    Alineacion por fila: stateless[i] + stateful[i]  (per-flow feature vector)
    Carpetas: Light_Attack, Light_Benign, Heavy_Attack, Heavy_Benign
    NO incluye la carpeta Benign/ pura, igual que Exp 1.
    """
    if verbose:
        print('\n🧬 Loading CIC dataset (Exp-1 style: row-aligned, 4 folders)...')

    combined_frames = []
    stats = {'attack': 0, 'benign': 0}

    for label_type, folder_path in CIC_PATHS_EXP4.items():
        if not os.path.exists(folder_path):
            if verbose:
                print(f'   ⚠️  Folder not found: {folder_path}')
            continue

        label    = 1 if 'Attack' in label_type else 0
        sl_files = glob.glob(os.path.join(folder_path, 'stateless_*.csv'))

        if verbose:
            print(f'   📂 {label_type}: {len(sl_files)} files')

        for sl_path in sl_files:
            try:
                df_sl = pd.read_csv(sl_path)
                if df_sl.empty:
                    continue

                filename = os.path.basename(sl_path)
                if 'light_benign' in filename:
                    sf_filename = filename.replace('stateless_features-light_benign',
                                                   'stateful_features-_light_benign')
                else:
                    sf_filename = filename.replace('stateless_', 'stateful_')

                sf_path = os.path.join(folder_path, sf_filename)

                if not os.path.exists(sf_path):
                    candidates = glob.glob(os.path.join(folder_path, 'stateful_*.csv'))
                    core_name  = filename.replace('stateless_features-', '').replace('.pcap.csv', '')
                    match      = next((c for c in candidates if core_name in c), None)
                    if match:
                        sf_path = match
                    else:
                        continue

                df_sf  = pd.read_csv(sf_path)
                min_len = min(len(df_sl), len(df_sf))
                if min_len == 0:
                    continue

                df_sl = df_sl.iloc[:min_len].reset_index(drop=True)
                df_sf = df_sf.iloc[:min_len].reset_index(drop=True)

                df_sl = df_sl.reindex(columns=FEATS_SL, fill_value=0)
                df_sf = df_sf.reindex(columns=FEATS_SF, fill_value=0)

                for c in FEATS_SL:
                    df_sl[c] = pd.to_numeric(df_sl[c], errors='coerce').fillna(0)
                for c in FEATS_SF:
                    df_sf[c] = pd.to_numeric(df_sf[c], errors='coerce').fillna(0)

                df_hybrid = pd.concat([df_sl[FEATS_SL], df_sf[FEATS_SF]], axis=1)
                df_hybrid['label'] = label
                combined_frames.append(df_hybrid)
                stats['attack' if label == 1 else 'benign'] += len(df_hybrid)

            except Exception as e:
                if verbose:
                    print(f'      ⚠️  Error {os.path.basename(sl_path)}: {e}')

    if not combined_frames:
        raise RuntimeError('❌ No CIC data loaded')

    df_final = pd.concat(combined_frames, ignore_index=True)

    if verbose:
        print(f'   ✅ Total: {len(df_final):,}  '
              f'Attack: {stats["attack"]:,}  Benign: {stats["benign"]:,}')

    X = df_final[ALL_FEATS].values
    y = df_final['label'].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    if verbose:
        print(f'   ✂️  Train: {len(X_train):,}  Test: {len(X_test):,}')
    return X_train, y_train, X_test, y_test

# =============================================================================
# FUNCTION: LOAD ARAGAT
# =============================================================================

def load_aragat_attacks(aragat_dir, verbose=True):
    if verbose:
        print('\n📂 Loading ARAGAT Attacks...')

    import re
    pattern     = r'stateless_features-bridge\.pcap_(\d+)\.csv'
    seeds_found = set()

    for filename in os.listdir(aragat_dir):
        match = re.search(pattern, filename)
        if match:
            seed = match.group(1)
            if os.path.exists(os.path.join(aragat_dir, f'stateful_features-bridge.pcap_{seed}.csv')):
                seeds_found.add(seed)

    seeds_sorted = sorted(seeds_found, key=lambda x: int(x))
    if verbose:
        print(f'   └─ Seeds: {", ".join(seeds_sorted)}')

    all_X, all_y = [], []

    for seed in seeds_sorted:
        try:
            # Cargar stateless
            df_sl = pd.read_csv(os.path.join(aragat_dir, f'stateless_features-bridge.pcap_{seed}.csv'))
            for c in FEATS_SL:
                df_sl[c] = pd.to_numeric(df_sl.get(c, 0), errors='coerce')

            # Agregar stateless a nivel de tunnel_id (media)
            agg_dict = {feat: 'mean' for feat in FEATS_SL if feat in df_sl.columns}
            if 'label' in df_sl.columns:
                agg_dict['label'] = 'first'
            df_sl_agg = df_sl.groupby('tunnel_id').agg(agg_dict).reset_index()
            df_sl_agg['tunnel_id'] = df_sl_agg['tunnel_id'].astype(str) + f'_s{seed}'

            # Cargar stateful
            df_sf = pd.read_csv(os.path.join(aragat_dir, f'stateful_features-bridge.pcap_{seed}.csv'))
            for c in FEATS_SF:
                df_sf[c] = pd.to_numeric(df_sf.get(c, 0), errors='coerce')
            df_sf['tunnel_id'] = df_sf['tunnel_id'].astype(str) + f'_s{seed}'

            df_merged = pd.merge(df_sl_agg, df_sf, on='tunnel_id', how='inner', suffixes=('', '_sf'))
            df_merged  = df_merged.reindex(columns=ALL_FEATS + ['label', 'tunnel_id'], fill_value=0)
            df_merged['label']    = df_merged['label'].fillna(1).astype(int)
            df_merged[ALL_FEATS]  = df_merged[ALL_FEATS].replace([np.inf, -np.inf], np.nan).fillna(0)

            df_attacks = df_merged[df_merged['label'] == 1]
            if len(df_attacks) > 0:
                all_X.append(df_attacks[ALL_FEATS].values)
                all_y.append(df_attacks['label'].values)
                if verbose:
                    print(f'      ├─ Seed {seed}: {len(df_attacks):,}')
        except Exception as e:
            if verbose:
                print(f'      ⚠️  {seed}: {e}')

    X = np.vstack(all_X)
    y = np.concatenate(all_y)
    if verbose:
        print(f'   ✅ Total: {len(X):,}')
    return X, y

# =============================================================================
# TRAINING AND EVALUATION
# =============================================================================

def train_ml(name, X, y):
    if name == 'RandomForest':
        m = RandomForestClassifier(n_estimators=100, max_depth=10,
                                   class_weight='balanced', random_state=42, n_jobs=-1)
    elif name == 'XGBoost':
        m = XGBClassifier(n_estimators=100, max_depth=10, learning_rate=0.1,
                          scale_pos_weight=(y==0).sum()/(y==1).sum(),
                          random_state=42, n_jobs=-1)
    elif name == 'LightGBM':
        m = LGBMClassifier(n_estimators=100, max_depth=10, learning_rate=0.1,
                           class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
    elif name == 'LogisticRegression':
        m = LogisticRegression(max_iter=1000, class_weight='balanced',
                               random_state=42, n_jobs=-1)
    m.fit(X, y)
    return m

def train_dl(name, X, y, cw, ep=20):
    inp = (X.shape[1], 1)
    if name == 'CNN':
        m = keras.Sequential([
            keras.layers.Conv1D(64, 3, activation='relu', input_shape=inp),
            keras.layers.MaxPooling1D(2),
            keras.layers.Conv1D(128, 3, activation='relu'),
            keras.layers.MaxPooling1D(2),
            keras.layers.Conv1D(256, 3, activation='relu'),
            keras.layers.GlobalAveragePooling1D(),
            keras.layers.Dense(128, activation='relu'),
            keras.layers.Dropout(0.5),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(1, activation='sigmoid')
        ])
    else:
        m = keras.Sequential([
            keras.layers.Bidirectional(keras.layers.LSTM(128, return_sequences=True),
                                       input_shape=inp),
            keras.layers.Dropout(0.3),
            keras.layers.Bidirectional(keras.layers.LSTM(64)),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(64, activation='relu'),
            keras.layers.Dropout(0.3),
            keras.layers.Dense(1, activation='sigmoid')
        ])
    m.compile(optimizer=keras.optimizers.Adam(0.001), loss='binary_crossentropy',
              metrics=['accuracy', keras.metrics.Recall(name='recall')])
    cb = [EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=0),
          ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                            min_lr=1e-7, verbose=0)]
    m.fit(X, y, epochs=ep, batch_size=64, validation_split=0.15,
          class_weight=cw, callbacks=cb, verbose=1)
    return m

def eval_model(m, X, y, dl=False):
    yp = (m.predict(X, verbose=0).flatten() > 0.5).astype(int) if dl else m.predict(X)
    return {
        'accuracy':  accuracy_score(y, yp),
        'precision': precision_score(y, yp, zero_division=0),
        'recall':    recall_score(y, yp, zero_division=0),
        'f1':        f1_score(y, yp, zero_division=0)
    }

# =============================================================================
# MAIN
# =============================================================================

if __name__ == '__main__':
    try:
        print('\n' + '='*70)
        print('🛡️  EXPERIMENT 4: DEFENSE HARDENING')
        print('='*70)

        # ── 1. CIC: same function as Exp 1 (row-by-row alignment, 4 folders) ──
        # With random_state=42 + stratify the split is identical to Exp 1,
        # so the baseline must reproduce exactly the same numbers.
        X_tr_cic, y_tr_cic, X_te_cic, y_te_cic = load_cic_exp1_style(verbose=True)

        # ── 2. ARAGAT: cargar todos los mutantes ──
        X_ar_all, y_ar_all = load_aragat_attacks(ARAGAT_DIR, verbose=True)

        # ── 3. Split ARAGAT 70 / 30 ──
        #    70% → hardening (visto en training)
        #    30% → held-out test (nunca visto)
        print('\n✂️  Splitting ARAGAT mutant data 70/30...')
        X_ar_train, X_ar_test, y_ar_train, y_ar_test = train_test_split(
            X_ar_all, y_ar_all, test_size=0.30, random_state=42
        )
        print(f'   ├─ Train (70%): {len(X_ar_train):,} mutant flows')
        print(f'   └─ Test  (30%): {len(X_ar_test):,} mutant flows  ← held-out')

        # ── 4. Hardened training set ──
        print('\n📦 Creating Hardened Training Set...')
        X_hard = np.vstack([X_tr_cic, X_ar_train])
        y_hard = np.concatenate([y_tr_cic, y_ar_train])
        print(f'   ├─ CIC train:           {len(X_tr_cic):,}')
        print(f'   ├─ ARAGAT train (70%):  {len(X_ar_train):,}')
        print(f'   └─ Hardened total:      {len(X_hard):,}  '
              f'({len(X_ar_train)/len(X_hard)*100:.2f}% adversarial)')

        # ── 5. Scaler (same one used to train baselines in Exp 1) ──
        scaler = joblib.load(os.path.join(MODEL_DIR_SOTA, 'scaler_sota.joblib'))
        print('\n✅ Scaler loaded (same as Experiment 1)')

        # ── 6. Escalar datos ──
        X_hard_sc    = scaler.transform(X_hard)
        X_hard_dl    = X_hard_sc.reshape((X_hard_sc.shape[0], X_hard_sc.shape[1], 1))

        X_te_sc      = scaler.transform(X_te_cic)
        X_te_dl      = X_te_sc.reshape((X_te_sc.shape[0], X_te_sc.shape[1], 1))

        X_ar_test_sc = scaler.transform(X_ar_test)
        X_ar_test_dl = X_ar_test_sc.reshape((X_ar_test_sc.shape[0], X_ar_test_sc.shape[1], 1))

        cw_arr  = compute_class_weight('balanced', classes=np.unique(y_hard), y=y_hard)
        cw_dict = {i: cw_arr[i] for i in range(len(cw_arr))}
        print(f'✅ Data prepared — class weights: {cw_dict}')

        # ── 7. Hardening loop ──
        results = []
        models  = [('RandomForest',      False),
                   ('XGBoost',           False),
                   ('LightGBM',          False),
                   ('LogisticRegression',False),
                   ('CNN',               True),
                   ('LSTM',              True)]

        for name, dl in models:
            print(f"\n{'='*70}\n⚙️  {name}\n{'='*70}")
            try:
                # Cargar modelo original (baseline de Exp 1)
                ext       = 'keras' if dl else 'joblib'
                orig_path = os.path.join(MODEL_DIR_SOTA, f'{name}_sota.{ext}')
                if not os.path.exists(orig_path):
                    print(f'   ⚠️  Model not found: {orig_path}')
                    continue
                m_orig = keras.models.load_model(orig_path) if dl else joblib.load(orig_path)
                print('   ✅ Baseline model loaded')

                # Datos por tipo de modelo
                if name == 'LogisticRegression':
                    Xtr, Xte_cic, Xte_mut = X_hard_sc,  X_te_sc,  X_ar_test_sc
                elif dl:
                    Xtr, Xte_cic, Xte_mut = X_hard_dl,  X_te_dl,  X_ar_test_dl
                else:
                    Xtr, Xte_cic, Xte_mut = X_hard,     X_te_cic, X_ar_test

                # Entrenar modelo hardened
                print('   🔧 Training hardened model...')
                m_hard = (train_dl(name, Xtr, y_hard, cw_dict, 20)
                          if dl else train_ml(name, Xtr, y_hard))

                # Guardar
                save_path = os.path.join(MODEL_DIR_HARDENED, f'{name}_hardened.{ext}')
                (m_hard.save(save_path) if dl else joblib.dump(m_hard, save_path))
                print(f'   💾 Saved: {save_path}')

                # Evaluation
                # CIC test set: same distribution as Exp 1 → baselines must match
                mo_cic = eval_model(m_orig, Xte_cic, y_te_cic, dl)
                mh_cic = eval_model(m_hard, Xte_cic, y_te_cic, dl)

                # Held-out 30% mutantes: nunca visto en training
                mo_mut = eval_model(m_orig, Xte_mut, y_ar_test, dl)
                mh_mut = eval_model(m_hard, Xte_mut, y_ar_test, dl)

                # Acumular
                for ds, mo, mh in [('CIC-2021',                    mo_cic, mh_cic),
                                    ('Mutant Payload (held-out 30%)', mo_mut, mh_mut)]:
                    results.append({'model': name, 'version': 'Baseline', 'dataset': ds, **mo})
                    results.append({'model': name, 'version': 'Hardened', 'dataset': ds, **mh})

                # Mostrar resumen
                print(f'\n   ✅ CIC-2021 — Stability (should match Exp 1 baseline):')
                print(f'      Recall  Baseline: {mo_cic["recall"]:.2%}  →  Hardened: {mh_cic["recall"]:.2%}  '
                      f'({mh_cic["recall"]-mo_cic["recall"]:+.2%})')
                print(f'      F1      Baseline: {mo_cic["f1"]:.2%}  →  Hardened: {mh_cic["f1"]:.2%}  '
                      f'({mh_cic["f1"]-mo_cic["f1"]:+.2%})')
                print(f'\n   ✅ Mutant held-out (30%) — Recovery:')
                print(f'      Recall  Baseline: {mo_mut["recall"]:.2%}  →  Hardened: {mh_mut["recall"]:.2%}  '
                      f'({mh_mut["recall"]-mo_mut["recall"]:+.2%})')
                print(f'      F1      Baseline: {mo_mut["f1"]:.2%}  →  Hardened: {mh_mut["f1"]:.2%}  '
                      f'({mh_mut["f1"]-mo_mut["f1"]:+.2%})')

            except Exception as e:
                print(f'   ❌ Error: {e}')
                import traceback; traceback.print_exc()

        # ── 8. Guardar resultados ──
        if results:
            df_res   = pd.DataFrame(results)
            csv_path = os.path.join(MODEL_DIR_HARDENED, 'experiment4_hardening_results.csv')
            df_res.to_csv(csv_path, index=False)

            print('\n' + '='*70)
            print('✅ EXPERIMENT 4 COMPLETE')
            print('='*70)
            print(f'\n📁 Results: {csv_path}')

            print('\n📊 CIC-2021 — Stability Check (Baseline vs Hardened):')
            df_cic = df_res[df_res['dataset'] == 'CIC-2021']
            print(df_cic[['model','version','recall','f1','precision','accuracy']].to_string(index=False))

            print('\n📊 Mutant Payload — Held-out 30% (Baseline vs Hardened):')
            df_mut = df_res[df_res['dataset'] == 'Mutant Payload (held-out 30%)']
            print(df_mut[['model','version','recall','f1','precision','accuracy']].to_string(index=False))
        else:
            print('\n❌ No models trained')

    except Exception as e:
        print(f'\n❌ ERROR: {e}')
        import traceback; traceback.print_exc()

## Experiment 5: Transfer Learning to Unknown Tools (GraphTunnel)

Evaluate baseline and hardened models on 8 DNS tunneling tools from the GraphTunnel dataset
that were **not present** in the CIC Bell DNS 2021 training set (zero-shot transfer).

Tools evaluated: dns2tcp, tcp-over-dns-CNAME, tcp-over-dns-TXT, dnspot, DNS-shell,
cobalstrike, ozymandns, tuns

In [ ]:
# =============================================================================
# EXPERIMENT 5: TRANSFER LEARNING TO UNKNOWN TOOLS
# Evaluation per individual tool — 8 canonical tools, no duplicates
# =============================================================================

import pandas as pd
import glob
import os
import re
import numpy as np
import math
import subprocess
import joblib
from collections import Counter
import sys
if 'tensorflow' not in sys.modules:
    import tensorflow as tf
else:
    tf = sys.modules['tensorflow']
keras = tf.keras
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    confusion_matrix
)

# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR           = '/content/drive/MyDrive/Tunnel/CSV_CIC21'
MODEL_DIR_SOTA     = os.path.join(BASE_DIR, 'Models_SOTA_Hybrid')
MODEL_DIR_HARDENED = os.path.join(BASE_DIR, 'Models_Hardened')
GT_DATA_DIR        = '/content/drive/MyDrive/Tunnel/GraphTunnel_Repo'

# Las mismas 23 features del resto del paper
FEATS_SL = [
    'FQDN_count', 'subdomain_length', 'upper', 'lower', 'numeric',
    'entropy', 'special', 'labels', 'labels_max', 'labels_average', 'len'
]
FEATS_SF = [
    'rr',
    'A_frequency', 'AAAA_frequency', 'CNAME_frequency', 'TXT_frequency',
    'MX_frequency', 'NS_frequency', 'NULL_frequency',
    'rr_count', 'distinct_ip', 'unique_ttl', 'total_queries'
]
ALL_FEATS = FEATS_SL + FEATS_SF

# =============================================================================
# HERRAMIENTAS CANONICAS — 8 tools, sin duplicados
# When a folder with multiple pcaps exists, use the folder (more data).
# Cuando solo existe un pcap individual, se usa ese archivo.
# =============================================================================

def resolve_tool_paths(base):
    # Returns dict mapping tool_name -> [list of pcap paths]
    def folder_pcaps(rel_folder):
        pattern = os.path.join(base, rel_folder, '*.pcap')
        return sorted(glob.glob(pattern))

    def single_pcap(rel_path):
        p = os.path.join(base, rel_path)
        return [p] if os.path.exists(p) else []

    return {
        'dns2tcp':            folder_pcaps('tunnel/dns2tcp-key'),
        'tcp-over-dns-CNAME': folder_pcaps('unkownTunnel/tcp-over-dns-CNAME'),
        'tcp-over-dns-TXT':   folder_pcaps('unkownTunnel/tcp-over-dns-TXT'),
        'dnspot':             folder_pcaps('tunnel/dnspot'),
        'DNS-shell':          single_pcap('tunnel/DNS-shell.pcap'),
        'cobalstrike':        single_pcap('unkownTunnel/cobalstrike.pcap'),
        'ozymandns':          single_pcap('unkownTunnel/ozymandns.pcap'),
        'tuns':               single_pcap('tunnel/tuns.pcap'),
    }

def resolve_benign_paths(base):
    return (sorted(glob.glob(os.path.join(base, 'normal', '**', '*.pcap'), recursive=True)) +
            sorted(glob.glob(os.path.join(base, 'wildcard', '*.pcap'))))

# =============================================================================
# FEATURE EXTRACTOR
# =============================================================================

class CICBellFeatureExtractor:
    def __init__(self):
        self.qtype_map = {
            1: 'A', 28: 'AAAA', 5: 'CNAME', 16: 'TXT',
            15: 'MX', 2: 'NS', 12: 'PTR', 10: 'NULL',
            6: 'SOA', 33: 'SRV'
        }

    def get_entropy(self, s):
        if not s:
            return 0.0
        cnt = Counter(s)
        n   = len(s)
        return -sum((c / n) * math.log2(c / n) for c in cnt.values())

    def extract_subdomain_features(self, qname):
        parts     = qname.split('.')
        subdomain = '.'.join(parts[:-2]) if len(parts) >= 2 else ''
        clean_sub = subdomain.replace('.', '')
        return {
            'FQDN_count':       len(qname),
            'subdomain_length': len(subdomain),
            'upper':    sum(c.isupper() for c in clean_sub),
            'lower':    sum(c.islower() for c in clean_sub),
            'numeric':  sum(c.isdigit() for c in clean_sub),
            'entropy':  self.get_entropy(clean_sub),
            'special':  sum(not c.isalnum() for c in clean_sub),
            'labels':        len(parts),
            'labels_max':    max(len(p) for p in parts) if parts else 0,
            'labels_average': np.mean([len(p) for p in parts]) if parts else 0,
            'len': len(qname)
        }

    def process_pcap(self, pcap_path, limit=3000):
        try:
            import scapy.all as scapy
        except ImportError:
            subprocess.run(['pip', 'install', 'scapy', '--quiet'], check=False)
            import scapy.all as scapy

        try:
            packets = scapy.rdpcap(pcap_path, count=limit)
        except Exception as e:
            print(f'         WARNING rdpcap error: {e}')
            return pd.DataFrame()

        stateless_rows = []
        qtype_counts   = Counter()
        ttl_values     = set()
        resolved_ips   = set()
        total_queries  = 0

        for pkt in packets:
            if pkt.haslayer(scapy.DNS) and pkt.haslayer(scapy.DNSQR):
                try:
                    qname = pkt[scapy.DNSQR].qname.decode('utf-8', 'ignore').rstrip('.')
                    if not qname:
                        continue
                    qtype = self.qtype_map.get(pkt[scapy.DNSQR].qtype, 'OTHER')
                    stateless_rows.append(self.extract_subdomain_features(qname))
                    qtype_counts[qtype] += 1
                    total_queries += 1
                    if pkt.haslayer(scapy.DNSRR):
                        for i in range(pkt[scapy.DNS].ancount):
                            try:
                                rr = pkt[scapy.DNS].an[i]
                                ttl_values.add(rr.ttl)
                                if rr.type in [1, 28] and hasattr(rr, 'rdata'):
                                    resolved_ips.add(str(rr.rdata))
                            except:
                                continue
                except:
                    continue

        if total_queries == 0:
            return pd.DataFrame()

        stateful = {
            'rr':              qtype_counts['A'] / max(qtype_counts['A'] + qtype_counts['AAAA'], 1),
            'A_frequency':     qtype_counts['A']     / total_queries,
            'AAAA_frequency':  qtype_counts['AAAA']  / total_queries,
            'CNAME_frequency': qtype_counts['CNAME'] / total_queries,
            'TXT_frequency':   qtype_counts['TXT']   / total_queries,
            'MX_frequency':    qtype_counts['MX']    / total_queries,
            'NS_frequency':    qtype_counts['NS']    / total_queries,
            'NULL_frequency':  qtype_counts['NULL']  / total_queries,
            'rr_count':        total_queries,
            'distinct_ip':     len(resolved_ips),
            'unique_ttl':      len(ttl_values),
            'total_queries':   total_queries
        }

        rows = [{**sl, **stateful} for sl in stateless_rows]
        return pd.DataFrame(rows)

# =============================================================================
# EVALUATION
# =============================================================================

def eval_model_transfer(m, X, y, is_dl=False, X_dl=None):
    if is_dl:
        y_pred = (m.predict(X_dl, verbose=0).flatten() > 0.5).astype(int)
    else:
        y_pred = m.predict(X)

    cm = confusion_matrix(y, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    else:
        fpr = 0.0

    return {
        'recall':    recall_score(y, y_pred, zero_division=0),
        'precision': precision_score(y, y_pred, zero_division=0),
        'f1':        f1_score(y, y_pred, zero_division=0),
        'accuracy':  accuracy_score(y, y_pred),
        'fpr':       fpr
    }

# =============================================================================
# MAIN
# =============================================================================

if __name__ == '__main__':
    try:
        print('\n' + '=' * 70)
        print('EXPERIMENT 5: TRANSFER LEARNING TO UNKNOWN TOOLS')
        print('=' * 70)

        # ── Setup repo ─────────────────────────────────────────────────────
        if not os.path.exists(GT_DATA_DIR):
            print('   Cloning GraphTunnel repo...')
            subprocess.run(
                ['git', 'clone', '--depth', '1',
                 'https://github.com/DNS-Datasets/GraphTunnel.git',
                 GT_DATA_DIR],
                check=True, capture_output=True
            )
        print(f'   Repo: {GT_DATA_DIR}')

        # ── STEP 1: Resolve canonical paths ──────────────────────────────
        tool_paths   = resolve_tool_paths(GT_DATA_DIR)
        benign_paths = resolve_benign_paths(GT_DATA_DIR)

        print('\n' + '=' * 70)
        print('PASO 1: HERRAMIENTAS CANONICAS (8 tools, sin duplicados)')
        print('=' * 70)
        for tool, paths in tool_paths.items():
            print(f'   [{tool}] -> {len(paths)} pcap(s)')
            for p in paths:
                print(f'       {os.path.relpath(p, GT_DATA_DIR)}')
        print(f'\n   Benign pcaps: {len(benign_paths)}')

        missing = [t for t, p in tool_paths.items() if not p]
        if missing:
            print(f'\n   WARNING -- tools with no pcaps found: {missing}')
            tool_paths = {t: p for t, p in tool_paths.items() if p}

        if not tool_paths:
            raise RuntimeError('No valid attack tools found -- check GT_DATA_DIR')

        # ── PASO 2: Extraer features ───────────────────────────────────────
        print('\n' + '=' * 70)
        print('PASO 2: EXTRACCION DE FEATURES')
        print('=' * 70)

        extractor = CICBellFeatureExtractor()

        # Benign — balanced subset (max 500 pcaps)
        n_benign = min(len(benign_paths), 500)
        np.random.seed(42)
        sel_benign = list(np.random.choice(benign_paths, n_benign, replace=False))

        print(f'\n   Extracting benign from {n_benign} pcaps...')
        benign_dfs = []
        for i, p in enumerate(sel_benign, 1):
            if i % 50 == 0:
                print(f'      [{i}/{n_benign}] {os.path.basename(p)}')
            df = extractor.process_pcap(p, limit=500)
            if not df.empty:
                df['label'] = 0
                benign_dfs.append(df)
        if not benign_dfs:
            raise RuntimeError('No benign data extracted')
        df_benign = pd.concat(benign_dfs, ignore_index=True)
        print(f'   Benign: {len(df_benign):,} samples')

        # Ataques — por herramienta
        tool_dfs = {}
        for tool, paths in tool_paths.items():
            print(f'\n   Tool: {tool} ({len(paths)} pcap(s))')
            frames = []
            for p in paths:
                print(f'      -> {os.path.basename(p)}', end=' ')
                df = extractor.process_pcap(p, limit=3000)
                if not df.empty:
                    df['label'] = 1
                    frames.append(df)
                    print(f'OK ({len(df):,} rows)')
                else:
                    print('empty')
            if frames:
                tool_dfs[tool] = pd.concat(frames, ignore_index=True)
                print(f'      TOTAL {tool}: {len(tool_dfs[tool]):,} samples')

        if not tool_dfs:
            raise RuntimeError('No attack data extracted')

        # ── PASO 3: Cargar modelos y evaluar ──────────────────────────────
        print('\n' + '=' * 70)
        print('PASO 3: EVALUACION POR HERRAMIENTA')
        print('=' * 70)

        scaler = joblib.load(os.path.join(MODEL_DIR_SOTA, 'scaler_sota.joblib'))
        print('Scaler loaded')

        MODELS = [
            ('LogisticRegression', False, 'joblib'),
            ('RandomForest',       False, 'joblib'),
            ('XGBoost',            False, 'joblib'),
            ('LightGBM',           False, 'joblib'),
            ('CNN',                True,  'keras'),
            ('LSTM',               True,  'keras'),
        ]

        def load_model(name, ext, version):
            path = (os.path.join(MODEL_DIR_SOTA,     f'{name}_sota.{ext}')
                    if version == 'Baseline' else
                    os.path.join(MODEL_DIR_HARDENED, f'{name}_hardened.{ext}'))
            if not os.path.exists(path):
                return None
            return keras.models.load_model(path) if ext == 'keras' else joblib.load(path)

        all_results = []

        for tool in sorted(tool_dfs):
            df_attack = tool_dfs[tool]
            print(f'\n{"─"*70}')
            print(f'   Tool: {tool}  ({len(df_attack):,} attack + {len(df_benign):,} benign)')
            print(f'   {"─"*66}')

            df_eval = pd.concat([df_attack, df_benign], ignore_index=True)
            X_raw   = df_eval.reindex(columns=ALL_FEATS, fill_value=0)
            X_raw   = X_raw.replace([np.inf, -np.inf], np.nan).fillna(0).values
            y_true  = df_eval['label'].values

            X_sc = scaler.transform(X_raw)
            X_dl = X_sc.reshape((X_sc.shape[0], X_sc.shape[1], 1))

            for name, is_dl, ext in MODELS:
                for version in ['Baseline', 'Hardened']:
                    m = load_model(name, ext, version)
                    if m is None:
                        print(f'      WARNING {version} {name} not found')
                        continue

                    X_in = X_sc if name == 'LogisticRegression' else X_raw
                    met  = eval_model_transfer(m, X_in, y_true,
                                               is_dl=is_dl,
                                               X_dl=X_dl if is_dl else None)
                    all_results.append({'tool': tool, 'model': name,
                                        'version': version, **met})
                    print(f'      [{version:8s}] {name:20s}  '
                          f'Recall={met["recall"]:.2%}  '
                          f'Prec={met["precision"]:.2%}  '
                          f'FPR={met["fpr"]:.2%}')

        if not all_results:
            raise RuntimeError('No results collected')

        df_res = pd.DataFrame(all_results)

        # ── Tabla detallada por herramienta ────────────────────────────────
        print('\n\n' + '=' * 70)
        print('RESULTADOS DETALLADOS POR HERRAMIENTA')
        print('=' * 70)

        for tool in sorted(df_res['tool'].unique()):
            df_t = df_res[df_res['tool'] == tool]
            print(f'\n  -- {tool} --')
            print(f'  {"Model":<22} {"Version":<10} {"Recall":>8} {"Precision":>10} {"FPR":>8} {"F1":>8}')
            print(f'  {"-"*66}')
            for _, r in df_t.sort_values(['model', 'version']).iterrows():
                print(f'  {r["model"]:<22} {r["version"]:<10} '
                      f'{r["recall"]:>8.2%} {r["precision"]:>10.2%} '
                      f'{r["fpr"]:>8.2%} {r["f1"]:>8.2%}')

        # ── Tabla resumen ──────────────────────────────────────────────────
        print('\n\n' + '=' * 70)
        print('TABLA RESUMEN -- PROMEDIO POR MODELO (8 herramientas)')
        print('=' * 70)
        print(f'  {"Model":<22} {"Version":<10} '
              f'{"Recall":>8} {"Precision":>10} {"FPR":>8} {"F1":>8}')
        print(f'  {"-"*66}')

        summary = (df_res
                   .groupby(['model', 'version'], sort=False)
                   [['recall', 'precision', 'fpr', 'f1']]
                   .mean()
                   .reset_index())

        model_order = ['LogisticRegression', 'RandomForest', 'XGBoost',
                       'LightGBM', 'CNN', 'LSTM']
        summary['model'] = pd.Categorical(summary['model'],
                                          categories=model_order, ordered=True)
        summary = summary.sort_values(['model', 'version'])

        for _, r in summary.iterrows():
            print(f'  {r["model"]:<22} {r["version"]:<10} '
                  f'{r["recall"]:>8.2%} {r["precision"]:>10.2%} '
                  f'{r["fpr"]:>8.2%} {r["f1"]:>8.2%}')

        # ── Guardar CSV ───────────────────────────────────────────────────
        csv_path = os.path.join(MODEL_DIR_HARDENED,
                                'experiment5_transfer_learning_results.csv')
        df_res.to_csv(csv_path, index=False)
        print(f'\nResults saved: {csv_path}')

        print('\n' + '=' * 70)
        print('EXPERIMENT 5 COMPLETE')
        print('=' * 70)

    except Exception as e:
        print(f'\nERROR: {e}')
        import traceback; traceback.print_exc()
